# 03. Model Training

이 Notebook에서는 전처리가 완료된 mmBERT Token Classification Dataset을 이용하여 민감정보 탐지 모델을 학습하고 평가한다.

이전 단계에서 다음 Pipeline을 완료하였다.

- Structure-aware Block Builder
- Block-local Character Span 변환
- mmBERT Tokenizer 적용
- Character Span → BIO Label Alignment
- `max_length=1024`, `stride=256` 정책 확정
- Train / Validation / Test Tokenized Dataset 생성
- Tokenized Dataset 무결성 검증

이번 Notebook에서는 다음 작업을 진행한다.

1. mmBERT Baseline 모델 구성
2. 학습 Dataset 및 Dynamic Padding 구성
3. Baseline Fine-tuning
4. Validation 평가
5. Block-level Character Span 기반 성능 평가
6. Test 평가
7. Error Analysis
8. 성능 개선

## 29. mmBERT Baseline Token Classification

Primary Backbone으로 선정한 `jhu-clsp/mmBERT-base`에 Token Classification Head를 추가하여 25개의 BIO Label을 예측하는 Baseline 모델을 구성한다.

Baseline에서는 복잡한 성능 개선 기법을 적용하지 않는다.

먼저 기본 Cross Entropy Loss를 이용하여 모델이 현재 Dataset에서 어느 정도의 성능을 보이는지 확인한다.

Padding은 모든 Sequence를 1024 Token으로 고정하지 않고 Batch 내부의 가장 긴 Sequence에 맞추는 Dynamic Padding을 사용한다.

`-100` Label은 Loss 계산에서 제외한다.

### 29.1 Tokenized Dataset 및 Manifest 로드

학습 단계에서는 전처리 Notebook에서 저장한 mmBERT 전용 Tokenized Dataset을 파일에서 다시 로드한다.

이를 통해 이전 Notebook의 Python 변수에 의존하지 않고 학습 과정을 독립적으로 재현할 수 있도록 한다.

사용하는 Dataset은 다음과 같다.

- Train
- Validation
- Test

Test Dataset은 현재 학습이나 Hyperparameter 결정에 사용하지 않는다.

In [4]:
import accelerate

from transformers.utils import (
    is_accelerate_available,
    ACCELERATE_MIN_VERSION,
)


is_accelerate_available.cache_clear()


print(
    "Accelerate version :",
    accelerate.__version__,
)

print(
    "Required version   :",
    ACCELERATE_MIN_VERSION,
)

print(
    "Available          :",
    is_accelerate_available(),
)


assert is_accelerate_available()

Accelerate version : 1.14.0
Required version   : 1.1.0
Available          : True


In [5]:
from pathlib import Path
from collections import Counter

import json
import os

import numpy as np
import pandas as pd

In [6]:
TOKENIZED_DIR = Path(
    "../data/processed/tokenized/mmbert"
)


TRAIN_PATH = (
    TOKENIZED_DIR
    / "train.jsonl"
)

VALIDATION_PATH = (
    TOKENIZED_DIR
    / "validation.jsonl"
)

TEST_PATH = (
    TOKENIZED_DIR
    / "test.jsonl"
)

MANIFEST_PATH = (
    TOKENIZED_DIR
    / "manifest.json"
)

In [8]:
def load_jsonl(path):
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                records.append(
                    json.loads(line)
                )

    return records

In [9]:
train_records = load_jsonl(
    TRAIN_PATH
)

validation_records = load_jsonl(
    VALIDATION_PATH
)

test_records = load_jsonl(
    TEST_PATH
)


with MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    manifest = json.load(f)


print(
    f"Train      : "
    f"{len(train_records):,}"
)

print(
    f"Validation : "
    f"{len(validation_records):,}"
)

print(
    f"Test       : "
    f"{len(test_records):,}"
)

print(
    f"Labels     : "
    f"{manifest['num_labels']}"
)

print(
    f"Backbone   : "
    f"{manifest['backbone']}"
)

Train      : 44,626
Validation : 5,163
Test       : 5,385
Labels     : 25
Backbone   : jhu-clsp/mmBERT-base


### 29.2 BIO Label Mapping 복원

Token Classification Head의 출력 순서와 Dataset Label ID가 정확히 일치해야 한다.

전처리 단계에서 저장한 Manifest의 `label2id`, `id2label`을 그대로 모델 Config에 적용한다.

In [10]:
label2id = (
    manifest[
        "label2id"
    ]
)


id2label = {
    int(key): value
    for key, value
    in manifest[
        "id2label"
    ].items()
}


NUM_LABELS = (
    manifest[
        "num_labels"
    ]
)

PRIMARY_BACKBONE = (
    manifest[
        "backbone"
    ]
)

MAX_LENGTH = (
    manifest[
        "max_length"
    ]
)

IGNORE_INDEX = (
    manifest[
        "ignore_index"
    ]
)


print(
    f"Labels     : {NUM_LABELS}"
)

print(
    f"Max length : {MAX_LENGTH}"
)

print(
    f"Ignore idx : {IGNORE_INDEX}"
)

Labels     : 25
Max length : 1024
Ignore idx : -100


### 29.3 PyTorch 학습 Device 확인

현재 실행 환경에서 사용할 Accelerator를 확인한다.

우선순위는 다음과 같다.

1. CUDA
2. Apple Silicon MPS
3. CPU

MacBook의 Apple Silicon에서 MPS가 활성화되어 있다면 GPU 가속을 이용할 수 있다.

In [11]:
import torch
import transformers


if torch.cuda.is_available():
    device = torch.device(
        "cuda"
    )

elif (
    hasattr(
        torch.backends,
        "mps",
    )
    and torch.backends.mps.is_available()
):
    device = torch.device(
        "mps"
    )

else:
    device = torch.device(
        "cpu"
    )


print(
    f"PyTorch version      : "
    f"{torch.__version__}"
)

print(
    f"Transformers version : "
    f"{transformers.__version__}"
)

print(
    f"Device               : "
    f"{device}"
)

print(
    f"CUDA available       : "
    f"{torch.cuda.is_available()}"
)

print(
    f"MPS available        : "
    f"{torch.backends.mps.is_available()}"
)

PyTorch version      : 2.13.0
Transformers version : 5.15.0
Device               : mps
CUDA available       : False
MPS available        : True


### 29.4 Token Classification Dataset 구성

저장된 Tokenized Record에는 학습에 필요한 값 외에도 Character Offset과 Metadata가 포함되어 있다.

Baseline 모델의 `forward()`에 직접 필요한 값만 Dataset에서 반환한다.

사용하는 값은 다음 세 가지이다.

- `input_ids`
- `attention_mask`
- `labels`

`offset_mapping`, `source_sample_id` 등의 Metadata는 이후 Character Span 기반 평가 단계에서 다시 사용한다.

In [12]:
from torch.utils.data import Dataset


class TokenClassificationDataset(
    Dataset
):
    def __init__(
        self,
        records,
    ):
        self.records = records

    def __len__(
        self,
    ):
        return len(
            self.records
        )

    def __getitem__(
        self,
        index,
    ):
        record = (
            self.records[
                index
            ]
        )

        return {
            "input_ids":
                record[
                    "input_ids"
                ],

            "attention_mask":
                record[
                    "attention_mask"
                ],

            "labels":
                record[
                    "labels"
                ],
        }

In [13]:
train_dataset = (
    TokenClassificationDataset(
        train_records
    )
)

validation_dataset = (
    TokenClassificationDataset(
        validation_records
    )
)

test_dataset = (
    TokenClassificationDataset(
        test_records
    )
)


print(
    f"Train Dataset      : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation Dataset : "
    f"{len(validation_dataset):,}"
)

print(
    f"Test Dataset       : "
    f"{len(test_dataset):,}"
)

Train Dataset      : 44,626
Validation Dataset : 5,163
Test Dataset       : 5,385


### 29.5 Dynamic Padding 구성

각 Sequence의 실제 길이는 서로 다르다.

따라서 모든 Sample을 `max_length=1024`로 Padding하지 않고, 각 Batch에서 가장 긴 Sequence에 맞춰 Dynamic Padding을 적용한다.

Token Classification에서는 Input뿐 아니라 Label도 동일한 길이로 Padding해야 한다.

Label Padding 값은 `-100`으로 설정하여 Loss 계산에서 제외한다.

In [14]:
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
)


tokenizer = (
    AutoTokenizer.from_pretrained(
        PRIMARY_BACKBONE,
        use_fast=True,
    )
)


data_collator = (
    DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        padding="longest",
        label_pad_token_id=IGNORE_INDEX,
        return_tensors="pt",
    )
)

In [15]:
sample_batch = (
    data_collator(
        [
            train_dataset[0],
            train_dataset[1],
        ]
    )
)


print(
    "BATCH SHAPES"
)

print("=" * 50)


for key, value in (
    sample_batch.items()
):
    print(
        f"{key:<20} "
        f"{tuple(value.shape)}"
    )

BATCH SHAPES
input_ids            (2, 135)
attention_mask       (2, 135)
labels               (2, 135)


### 29.7 mmBERT Token Classification 모델 구성

사전학습된 mmBERT Encoder에 새로운 Token Classification Head를 추가한다.

출력 Label 수는 BIO Label 25개로 설정한다.

Encoder는 사전학습된 Weight를 사용하지만 Token Classification Head는 현재 프로젝트의 Label 수에 맞춰 새롭게 초기화된다.

따라서 모델 로드 시 Classification Head 일부 Weight가 새로 초기화되었다는 Warning이 나타나는 것은 정상이다.

In [16]:
from transformers import (
    AutoModelForTokenClassification,
)


model = (
    AutoModelForTokenClassification
    .from_pretrained(
        PRIMARY_BACKBONE,

        num_labels=NUM_LABELS,

        id2label=id2label,

        label2id=label2id,
    )
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)


print(
    f"Model class          : "
    f"{type(model).__name__}"
)

print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)

print(
    f"Number of labels     : "
    f"{model.config.num_labels}"
)

Model class          : ModernBertForTokenClassification
Total parameters     : 307,549,465
Trainable parameters : 307,549,465
Number of labels     : 25


### 29.8 Forward Pass Smoke Test

전체 Fine-tuning을 시작하기 전에 하나의 작은 Batch를 모델에 입력하여 학습 Pipeline이 정상적으로 연결되는지 확인한다.

다음 항목을 검증한다.

- Device 이동
- Input Shape
- Logit Shape
- Loss 계산
- BIO Label 25개 출력

이 단계가 정상적으로 동작한 뒤 실제 TrainingArguments와 Trainer를 구성한다.

In [18]:
smoke_index = min(
    range(
        len(train_records)
    ),
    key=lambda index:
        len(
            train_records[
                index
            ]["input_ids"]
        ),
)


smoke_batch = (
    data_collator(
        [
            train_dataset[
                smoke_index
            ]
        ]
    )
)


print(
    "Smoke sequence length:",
    smoke_batch[
        "input_ids"
    ].shape[1],
)

Smoke sequence length: 13


In [19]:
model = model.to(
    device
)

model.eval()


device_batch = {
    key: value.to(
        device
    )
    for key, value
    in smoke_batch.items()
}


with torch.no_grad():
    outputs = model(
        **device_batch
    )


print(
    f"Device       : "
    f"{device}"
)

print(
    f"Loss         : "
    f"{outputs.loss.item():.6f}"
)

print(
    f"Logits shape : "
    f"{tuple(outputs.logits.shape)}"
)

Device       : mps
Loss         : 4.428921
Logits shape : (1, 13, 25)


### 29.9 Baseline 학습 정책

전체 학습에 앞서 Baseline Fine-tuning 설정을 정의한다.

Baseline의 목적은 복잡한 최적화 없이 현재 Dataset과 mmBERT가 어느 정도의 성능을 보이는지 기준 성능을 확보하는 것이다.

초기 학습 정책은 다음과 같다.

- Epoch: 3
- Learning Rate: 3e-5
- Train Batch Size: 2
- Gradient Accumulation: 8
- Effective Batch Size: 16
- Validation Batch Size: 2
- Optimizer: AdamW
- Weight Decay: 0.01
- Warmup Ratio: 0.1
- LR Scheduler: Linear
- Best Model 기준: Validation Loss
- Dynamic Padding 사용
- Mixed Precision 미사용
- Gradient Checkpointing 미사용

Apple Silicon MPS에서 최대 1024 Token의 Sequence를 처리해야 하므로 실제 Device Batch Size는 보수적으로 2부터 시작한다.

Gradient Accumulation을 이용해 모델 Weight Update 기준의 Effective Batch Size는 16으로 유지한다.

Baseline 단계에서는 안정성과 재현성을 우선하며, Batch Size, Mixed Precision, Gradient Checkpointing 등의 최적화는 이후 성능 및 학습 효율 개선 단계에서 비교한다.

In [20]:
from transformers import TrainingArguments


MODEL_OUTPUT_DIR = (
    "../models/mmbert_baseline"
)


training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,

    # -------------------------
    # Training
    # -------------------------
    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=3e-5,
    weight_decay=0.01,

    # Transformers 5.15.0:
    # 0~1 사이 float는 전체 step 대비 비율
    warmup_steps=0.1,

    lr_scheduler_type="linear",

    max_grad_norm=1.0,

    # -------------------------
    # Optimizer
    # -------------------------
    optim="adamw_torch",

    # -------------------------
    # Evaluation / Checkpoint
    # -------------------------
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    # -------------------------
    # Logging
    # -------------------------
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,

    report_to="none",

    # -------------------------
    # Reproducibility
    # -------------------------
    seed=42,
    data_seed=42,

    # -------------------------
    # MPS
    # -------------------------
    fp16=False,
    bf16=False,

    dataloader_pin_memory=False,

    torch_empty_cache_steps=50,
)

### 29.11 Baseline 학습 설정 확인

정의한 Baseline 학습 설정이 의도한 값으로 적용되었는지 확인한다.

Apple Silicon 환경에서는 실제 Device가 `mps`로 설정되었는지도 함께 확인한다.

Baseline에서는 Device Batch Size를 2로 설정하고 Gradient Accumulation을 8회 적용하여 Effective Batch Size 16으로 학습한다.

In [21]:
effective_batch_size = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)


print(
    "BASELINE TRAINING CONFIG"
)

print("=" * 60)

print(
    f"Epochs                  : "
    f"{training_args.num_train_epochs}"
)

print(
    f"Train batch size        : "
    f"{training_args.per_device_train_batch_size}"
)

print(
    f"Gradient accumulation   : "
    f"{training_args.gradient_accumulation_steps}"
)

print(
    f"Effective batch size    : "
    f"{effective_batch_size}"
)

print(
    f"Eval batch size         : "
    f"{training_args.per_device_eval_batch_size}"
)

print(
    f"Learning rate           : "
    f"{training_args.learning_rate}"
)

print(
    f"Weight decay            : "
    f"{training_args.weight_decay}"
)

print(
    f"Warmup                  : "
    f"{training_args.warmup_steps}"
)

print(
    f"Optimizer               : "
    f"{training_args.optim}"
)

print(
    f"Device                  : "
    f"{training_args.device}"
)

BASELINE TRAINING CONFIG
Epochs                  : 3
Train batch size        : 2
Gradient accumulation   : 8
Effective batch size    : 16
Eval batch size         : 2
Learning rate           : 3e-05
Weight decay            : 0.01
Warmup                  : 0.1
Optimizer               : OptimizerNames.ADAMW_TORCH
Device                  : mps


### 29.12 Trainer Smoke Test Dataset 구성

Forward Pass는 정상적으로 동작했지만 실제 Fine-tuning에는 Backward Pass와 Optimizer Update가 포함된다.

전체 Dataset을 학습하기 전에 작은 Dataset으로 10 Step의 Smoke Test를 수행한다.

Smoke Test에서는 학습 Pipeline 자체의 안정성을 확인하는 것이 목적이므로 비교적 짧은 Sequence를 우선 사용한다.

확인할 항목은 다음과 같다.

- MPS Backward Pass
- AdamW Optimizer
- Gradient Accumulation
- Dynamic Padding
- Trainer Training Loop
- Loss 계산

In [23]:
from torch.utils.data import Subset


SMOKE_TRAIN_SIZE = 128


shortest_train_indices = sorted(
    range(
        len(train_records)
    ),
    key=lambda index:
        len(
            train_records[
                index
            ]["input_ids"]
        ),
)[:SMOKE_TRAIN_SIZE]


smoke_train_dataset = Subset(
    train_dataset,
    shortest_train_indices,
)


smoke_lengths = [
    len(
        train_records[index][
            "input_ids"
        ]
    )
    for index
    in shortest_train_indices
]


print(
    f"Smoke samples       : "
    f"{len(smoke_train_dataset):,}"
)

print(
    f"Min sequence length : "
    f"{min(smoke_lengths)}"
)

print(
    f"Max sequence length : "
    f"{max(smoke_lengths)}"
)

Smoke samples       : 128
Min sequence length : 13
Max sequence length : 14


### 29.13 Smoke Test 모델 구성

Smoke Test에서 Weight Update가 발생하므로 실제 Baseline 학습에 사용할 모델과 별도의 모델을 생성한다.

Smoke Test가 끝난 뒤 해당 모델은 폐기하고 실제 Baseline 학습에서는 사전학습된 mmBERT를 다시 로드한다.

In [24]:
from transformers import (
    AutoModelForTokenClassification,
)


smoke_model = (
    AutoModelForTokenClassification
    .from_pretrained(
        PRIMARY_BACKBONE,

        num_labels=NUM_LABELS,

        id2label=id2label,
        label2id=label2id,
    )
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
smoke_training_args = TrainingArguments(
    output_dir=(
        "../models/mmbert_smoke_test"
    ),

    max_steps=10,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=3e-5,

    optim="adamw_torch",

    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,

    eval_strategy="no",
    save_strategy="no",

    report_to="none",

    seed=42,

    fp16=False,
    bf16=False,

    dataloader_pin_memory=False,

    torch_empty_cache_steps=5,
)

In [26]:
from transformers import Trainer


smoke_trainer = Trainer(
    model=smoke_model,

    args=smoke_training_args,

    train_dataset=smoke_train_dataset,

    data_collator=data_collator,
)

### 29.16 Trainer Training Smoke Test

구성한 Smoke Test Dataset과 별도 mmBERT 모델을 이용하여 실제 10 Step Fine-tuning을 수행한다.

이 단계에서는 모델 성능 자체를 평가하지 않는다.

10 Step이 MPS 오류, Out Of Memory, NaN Loss 없이 완료되면 전체 Baseline 학습 Pipeline이 정상적으로 동작한다고 판단한다.

In [27]:
smoke_train_result = (
    smoke_trainer.train()
)

Step,Training Loss
1,8.108091
2,0.665885
3,0.004773
4,0.000057
5,0.000001
6,0.000001
7,0.000000
8,0.000000
9,0.000000
10,0.000000


In [28]:
print(
    "TRAINING SMOKE TEST RESULT"
)

print("=" * 60)

print(
    f"Training loss : "
    f"{smoke_train_result.training_loss:.6f}"
)

print(
    f"Global steps  : "
    f"{smoke_train_result.global_step}"
)

TRAINING SMOKE TEST RESULT
Training loss : 0.877881
Global steps  : 10


### 29.17 Smoke Test 정리

10 Step Training Smoke Test가 정상적으로 완료되었다.

이를 통해 다음 항목이 현재 Apple Silicon MPS 환경에서 정상적으로 동작함을 확인하였다.

- mmBERT Forward / Backward Pass
- AdamW Optimizer
- Gradient Accumulation
- Dynamic Padding
- Hugging Face Trainer
- Token Classification Loss 계산

Smoke Test 모델은 실제 Baseline 모델과 분리되어 있으므로 제거하고 MPS Memory를 정리한 뒤 새로운 사전학습 mmBERT 모델로 전체 학습을 시작한다.

In [29]:
import gc


del smoke_trainer
del smoke_model

gc.collect()


if torch.backends.mps.is_available():
    torch.mps.empty_cache()


print(
    "Smoke test resources cleared."
)

Smoke test resources cleared.


### 29.18 Baseline 모델 초기화

Smoke Test에서 Weight가 업데이트된 모델을 재사용하지 않는다.

동일한 사전학습 mmBERT Backbone에서 새로운 Token Classification 모델을 다시 로드하여 실제 Baseline Fine-tuning을 시작한다.

Token Classification Head는 25개의 BIO Label을 예측하도록 구성한다.

In [30]:
baseline_model = (
    AutoModelForTokenClassification
    .from_pretrained(
        PRIMARY_BACKBONE,

        num_labels=NUM_LABELS,

        id2label=id2label,
        label2id=label2id,
    )
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 29.19 Baseline Trainer 구성

전체 Train Dataset과 Validation Dataset을 사용하여 Baseline Trainer를 구성한다.

Validation은 Epoch마다 수행하며 `eval_loss`가 가장 낮은 Checkpoint를 Best Model로 선택한다.

Test Dataset은 학습 및 모델 선택에 사용하지 않는다.

In [31]:
baseline_trainer = Trainer(
    model=baseline_model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=validation_dataset,

    data_collator=data_collator,
)

In [32]:
import math


train_batches_per_epoch = math.ceil(
    len(train_dataset)
    / training_args.per_device_train_batch_size
)


optimizer_steps_per_epoch = math.ceil(
    train_batches_per_epoch
    / training_args.gradient_accumulation_steps
)


total_optimizer_steps = (
    optimizer_steps_per_epoch
    * int(
        training_args.num_train_epochs
    )
)


print(
    "BASELINE TRAINING SIZE"
)

print("=" * 60)

print(
    f"Train samples             : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation samples        : "
    f"{len(validation_dataset):,}"
)

print(
    f"Batches / epoch           : "
    f"{train_batches_per_epoch:,}"
)

print(
    f"Optimizer steps / epoch   : "
    f"{optimizer_steps_per_epoch:,}"
)

print(
    f"Total optimizer steps     : "
    f"{total_optimizer_steps:,}"
)

BASELINE TRAINING SIZE
Train samples             : 44,626
Validation samples        : 5,163
Batches / epoch           : 22,313
Optimizer steps / epoch   : 2,790
Total optimizer steps     : 8,370


### 29.21 mmBERT Baseline Fine-tuning

전체 Train Dataset을 이용하여 mmBERT Token Classification 모델을 3 Epoch Fine-tuning한다.

학습 중 Epoch마다 Validation Loss를 계산하고 가장 낮은 Validation Loss를 기록한 Checkpoint를 Best Model로 선택한다.

현재 단계에서는 Baseline 성능 확보가 목적이므로 별도의 Hyperparameter Tuning은 수행하지 않는다.

In [34]:
baseline_train_result = (
    baseline_trainer.train()
)

Epoch,Training Loss,Validation Loss
1,0.000001,0.000000
2,0.000001,0.000006
3,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
print(
    "BASELINE TRAINING RESULT"
)

print("=" * 60)

print(
    f"Training loss   : "
    f"{baseline_train_result.training_loss:.10f}"
)

print(
    f"Global steps    : "
    f"{baseline_train_result.global_step:,}"
)

print(
    f"Best checkpoint : "
    f"{baseline_trainer.state.best_model_checkpoint}"
)

print(
    f"Best metric     : "
    f"{baseline_trainer.state.best_metric}"
)

BASELINE TRAINING RESULT
Training loss   : 0.0041260482
Global steps    : 8,370
Best checkpoint : ../models/mmbert_baseline/checkpoint-8370
Best metric     : 1.7217381298451073e-08


In [36]:
from collections import Counter


validation_label_counts = Counter()

for record in validation_records:
    for label_id in record["labels"]:
        validation_label_counts[label_id] += 1


print(
    "VALIDATION LABEL DISTRIBUTION"
)

print("=" * 60)


for label_id, count in sorted(
    validation_label_counts.items()
):
    label_name = (
        "IGNORE"
        if label_id == IGNORE_INDEX
        else id2label[label_id]
    )

    print(
        f"{label_name:<30} "
        f"{count:>12,}"
    )

VALIDATION LABEL DISTRIBUTION
IGNORE                               10,326
O                                   627,892
B-EMAIL                                  86
I-EMAIL                                 595
B-PHONE                                  25
I-PHONE                                 329
B-PERSON                                 90
I-PERSON                                149
B-IP_ADDRESS                          1,993
I-IP_ADDRESS                         26,370
B-TOKEN                                 562
I-TOKEN                              25,057
B-API_KEY                               174
I-API_KEY                             4,385
B-SESSION_ID                            561
I-SESSION_ID                         16,185
B-PASSWORD                              213
I-PASSWORD                            3,421
B-SECRET                                155
I-SECRET                              5,240
B-PRIVATE_KEY                            32
I-PRIVATE_KEY                         3,097
B-

In [37]:
import torch


baseline_model = (
    baseline_trainer.model
)

baseline_model.eval()


sample_index = 0

sample = validation_dataset[
    sample_index
]


batch = data_collator(
    [sample]
)

batch = {
    key: value.to(device)
    for key, value
    in batch.items()
}


with torch.no_grad():
    outputs = baseline_model(
        input_ids=batch[
            "input_ids"
        ],
        attention_mask=batch[
            "attention_mask"
        ],
    )


pred_ids = (
    outputs.logits
    .argmax(dim=-1)[0]
    .cpu()
    .tolist()
)


gold_ids = (
    batch["labels"][0]
    .cpu()
    .tolist()
)


tokens = tokenizer.convert_ids_to_tokens(
    batch["input_ids"][0]
    .cpu()
    .tolist()
)


for token, gold_id, pred_id in zip(
    tokens,
    gold_ids,
    pred_ids,
):
    if gold_id == IGNORE_INDEX:
        continue

    gold_label = id2label[
        gold_id
    ]

    pred_label = id2label[
        pred_id
    ]

    if (
        gold_label != "O"
        or pred_label != "O"
    ):
        print(
            f"{token:<25} "
            f"Gold={gold_label:<25} "
            f"Pred={pred_label}"
        )

▁                         Gold=B-IP_ADDRESS              Pred=B-IP_ADDRESS
1                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
9                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
2                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
.                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
0                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
.                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
2                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
.                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
2                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS
0                         Gold=I-IP_ADDRESS              Pred=I-IP_ADDRESS


## 30. Validation Evaluation

Baseline 학습이 완료된 후 Validation Dataset을 이용하여 실제 민감정보 탐지 성능을 평가한다.

학습 과정에서는 Validation Loss를 기준으로 Best Checkpoint를 선택했지만, Loss만으로는 민감정보 탐지 성능을 직접 해석하기 어렵다.

따라서 다음 지표를 추가로 확인한다.

- Token-level Precision
- Token-level Recall
- Token-level F1-score
- Entity별 Precision / Recall / F1-score

`-100`으로 설정된 Special Token과 Padding Token은 평가에서 제외한다.

또한 전체 Token의 대부분을 차지하는 `O` Label의 영향으로 성능이 과대평가되지 않도록, 민감정보 Entity Token만을 대상으로 한 성능을 별도로 계산한다.

이후 BIO Prediction을 원문의 Character Span으로 복원하여 실제 마스킹 관점의 Span-level 평가를 수행한다.

In [38]:
best_model = baseline_trainer.model

best_model.eval()


print(
    "Best checkpoint :",
    baseline_trainer.state.best_model_checkpoint,
)

print(
    "Best metric     :",
    baseline_trainer.state.best_metric,
)

print(
    "Device          :",
    next(
        best_model.parameters()
    ).device,
)

Best checkpoint : ../models/mmbert_baseline/checkpoint-8370
Best metric     : 1.7217381298451073e-08
Device          : mps:0


### 30.3 Validation Prediction 준비

Validation Dataset 전체에 대해 Batch 단위 추론을 수행한다.

Dynamic Padding을 그대로 적용하며, 각 Sample의 원래 Token 길이에 맞추어 Prediction을 다시 분리하여 저장한다.

이 Prediction은 이후 Token-level 평가뿐 아니라 BIO Label을 Character Span으로 복원하는 과정에서도 재사용한다.

In [39]:
from torch.utils.data import DataLoader


VALIDATION_BATCH_SIZE = 2


validation_loader = DataLoader(
    validation_dataset,

    batch_size=VALIDATION_BATCH_SIZE,

    shuffle=False,

    collate_fn=data_collator,
)


print(
    f"Validation samples : "
    f"{len(validation_dataset):,}"
)

print(
    f"Validation batches : "
    f"{len(validation_loader):,}"
)

Validation samples : 5,163
Validation batches : 2,582


In [40]:
from tqdm.auto import tqdm

import torch


validation_pred_ids = []

best_model.eval()


with torch.no_grad():

    for batch in tqdm(
        validation_loader,
        desc="Validation inference",
    ):

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )


        outputs = best_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


        batch_predictions = (
            outputs.logits
            .argmax(dim=-1)
            .cpu()
            .tolist()
        )


        batch_attention = (
            attention_mask
            .cpu()
            .tolist()
        )


        for predictions, mask in zip(
            batch_predictions,
            batch_attention,
        ):

            sequence_length = sum(
                mask
            )


            validation_pred_ids.append(
                predictions[
                    :sequence_length
                ]
            )

Validation inference:   0%|          | 0/2582 [00:00<?, ?it/s]

In [41]:
prediction_count_error = 0
prediction_length_error = 0


if (
    len(validation_pred_ids)
    != len(validation_records)
):
    prediction_count_error += 1


for record, pred_ids in zip(
    validation_records,
    validation_pred_ids,
):

    if (
        len(record["input_ids"])
        != len(pred_ids)
    ):
        prediction_length_error += 1


print(
    "VALIDATION PREDICTION VALIDATION"
)

print("=" * 60)

print(
    f"Records                  : "
    f"{len(validation_records):,}"
)

print(
    f"Prediction records       : "
    f"{len(validation_pred_ids):,}"
)

print(
    f"Prediction count errors  : "
    f"{prediction_count_error:,}"
)

print(
    f"Prediction length errors : "
    f"{prediction_length_error:,}"
)

VALIDATION PREDICTION VALIDATION
Records                  : 5,163
Prediction records       : 5,163
Prediction count errors  : 0
Prediction length errors : 0


In [42]:
gold_label_ids = []
pred_label_ids = []


for record, predictions in zip(
    validation_records,
    validation_pred_ids,
):

    for gold_id, pred_id in zip(
        record["labels"],
        predictions,
    ):

        if gold_id == IGNORE_INDEX:
            continue


        gold_label_ids.append(
            gold_id
        )

        pred_label_ids.append(
            pred_id
        )


print(
    "TOKEN EVALUATION DATA"
)

print("=" * 60)

print(
    f"Evaluated tokens : "
    f"{len(gold_label_ids):,}"
)

TOKEN EVALUATION DATA
Evaluated tokens : 748,481


In [43]:
from sklearn.metrics import (
    accuracy_score,
)


token_accuracy = accuracy_score(
    gold_label_ids,
    pred_label_ids,
)


print(
    "TOKEN ACCURACY"
)

print("=" * 60)

print(
    f"Accuracy : "
    f"{token_accuracy:.6f}"
)

TOKEN ACCURACY
Accuracy : 1.000000


### 30.8 Entity Token Detection 성능

전체 Token Accuracy는 `O` Label 비중의 영향을 크게 받을 수 있다.

따라서 각 Token을 다음과 같이 이진화하여 민감정보 탐지 자체의 성능을 확인한다.

- Entity Token: 모든 `B-*`, `I-*` Label
- Non-Entity Token: `O`

이를 통해 모델이 민감정보 위치를 얼마나 정확하게 탐지하는지 Precision, Recall, F1-score로 평가한다.

In [44]:
from sklearn.metrics import (
    precision_recall_fscore_support,
)


O_LABEL_ID = label2id[
    "O"
]


gold_entity_binary = [
    int(
        label_id != O_LABEL_ID
    )
    for label_id
    in gold_label_ids
]


pred_entity_binary = [
    int(
        label_id != O_LABEL_ID
    )
    for label_id
    in pred_label_ids
]


(
    entity_precision,
    entity_recall,
    entity_f1,
    _,
) = precision_recall_fscore_support(
    gold_entity_binary,
    pred_entity_binary,

    average="binary",

    zero_division=0,
)


print(
    "ENTITY TOKEN DETECTION"
)

print("=" * 60)

print(
    f"Precision : "
    f"{entity_precision:.6f}"
)

print(
    f"Recall    : "
    f"{entity_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{entity_f1:.6f}"
)

ENTITY TOKEN DETECTION
Precision : 1.000000
Recall    : 1.000000
F1-score  : 1.000000


In [45]:
from sklearn.metrics import (
    classification_report,
)


label_ids = list(
    range(
        NUM_LABELS
    )
)


label_names = [
    id2label[label_id]
    for label_id
    in label_ids
]


bio_report = classification_report(
    gold_label_ids,
    pred_label_ids,

    labels=label_ids,
    target_names=label_names,

    digits=4,

    zero_division=0,
)


print(
    "BIO TOKEN CLASSIFICATION REPORT"
)

print("=" * 80)

print(
    bio_report
)

BIO TOKEN CLASSIFICATION REPORT
                     precision    recall  f1-score   support

                  O     1.0000    1.0000    1.0000    627892
            B-EMAIL     1.0000    1.0000    1.0000        86
            I-EMAIL     1.0000    1.0000    1.0000       595
            B-PHONE     1.0000    1.0000    1.0000        25
            I-PHONE     1.0000    1.0000    1.0000       329
           B-PERSON     1.0000    1.0000    1.0000        90
           I-PERSON     1.0000    1.0000    1.0000       149
       B-IP_ADDRESS     1.0000    1.0000    1.0000      1993
       I-IP_ADDRESS     1.0000    1.0000    1.0000     26370
            B-TOKEN     1.0000    1.0000    1.0000       562
            I-TOKEN     1.0000    1.0000    1.0000     25057
          B-API_KEY     1.0000    1.0000    1.0000       174
          I-API_KEY     1.0000    1.0000    1.0000      4385
       B-SESSION_ID     1.0000    1.0000    1.0000       561
       I-SESSION_ID     1.0000    1.0000    1.0000  

In [47]:
def bio_to_entity_type(
    label_name,
):

    if label_name == "O":
        return "O"

    return label_name.split(
        "-",
        1,
    )[1]


gold_entity_types = [
    bio_to_entity_type(
        id2label[label_id]
    )
    for label_id
    in gold_label_ids
]


pred_entity_types = [
    bio_to_entity_type(
        id2label[label_id]
    )
    for label_id
    in pred_label_ids
]


entity_type_labels = sorted(
    {
        label_name.split(
            "-",
            1,
        )[1]
        for label_name
        in label2id.keys()
        if label_name != "O"
    }
)


entity_type_report = (
    classification_report(
        gold_entity_types,
        pred_entity_types,

        labels=entity_type_labels,

        digits=4,

        zero_division=0,
    )
)


print(
    "ENTITY TYPE TOKEN REPORT"
)

print("=" * 80)

print(
    entity_type_report
)

ENTITY TYPE TOKEN REPORT
                   precision    recall  f1-score   support

          API_KEY     1.0000    1.0000    1.0000      4559
CONNECTION_STRING     1.0000    1.0000    1.0000      6925
            EMAIL     1.0000    1.0000    1.0000       681
       IP_ADDRESS     1.0000    1.0000    1.0000     28363
      PARAM_VALUE     1.0000    1.0000    1.0000     24945
         PASSWORD     1.0000    1.0000    1.0000      3634
           PERSON     1.0000    1.0000    1.0000       239
            PHONE     1.0000    1.0000    1.0000       354
      PRIVATE_KEY     1.0000    1.0000    1.0000      3129
           SECRET     1.0000    1.0000    1.0000      5395
       SESSION_ID     1.0000    1.0000    1.0000     16746
            TOKEN     1.0000    1.0000    1.0000     25619

        micro avg     1.0000    1.0000    1.0000    120589
        macro avg     1.0000    1.0000    1.0000    120589
     weighted avg     1.0000    1.0000    1.0000    120589



## 31. Character Span Reconstruction

Token Classification 모델은 각 Token에 대해 BIO Label을 예측한다.

그러나 최종 목표는 Token Label 자체가 아니라 원문에서 민감정보가 위치한 정확한 Character 구간을 탐지하고 마스킹하는 것이다.

따라서 Validation Prediction의 BIO Label과 `offset_mapping`을 이용하여 다음 형태의 Character Span으로 복원한다.

- `start`: 민감정보 시작 Character 위치
- `end`: 민감정보 종료 Character 위치
- `label`: Entity Type

복원된 Prediction Span을 원본 Gold Span과 비교하여 실제 마스킹 관점의 성능을 평가한다.

In [48]:
token_mismatch_count = 0


for gold_id, pred_id in zip(
    gold_label_ids,
    pred_label_ids,
):
    if gold_id != pred_id:
        token_mismatch_count += 1


print(
    "TOKEN PREDICTION SANITY CHECK"
)

print("=" * 60)

print(
    f"Evaluated tokens : "
    f"{len(gold_label_ids):,}"
)

print(
    f"Mismatches       : "
    f"{token_mismatch_count:,}"
)

print(
    f"Match ratio      : "
    f"{1 - token_mismatch_count / len(gold_label_ids):.8f}"
)

TOKEN PREDICTION SANITY CHECK
Evaluated tokens : 748,481
Mismatches       : 0
Match ratio      : 1.00000000


### 31.3 BIO Label을 Character Span으로 변환

각 Token의 `offset_mapping`을 사용하여 연속된 `B-ENTITY`, `I-ENTITY` Token을 하나의 Character Span으로 결합한다.

Special Token은 `(0, 0)` Offset을 가지므로 제외한다.

잘못된 BIO Sequence에서 `I-*`가 단독으로 등장하는 경우에는 해당 위치를 새로운 Entity 시작으로 처리하여 복원 과정이 중단되지 않도록 한다.

In [49]:
def decode_bio_to_spans(
    label_ids,
    offsets,
    id2label,
    ignore_index=-100,
):
    spans = []

    current_label = None
    current_start = None
    current_end = None


    def close_current_span():
        nonlocal current_label
        nonlocal current_start
        nonlocal current_end

        if current_label is not None:
            spans.append(
                {
                    "start": current_start,
                    "end": current_end,
                    "label": current_label,
                }
            )

        current_label = None
        current_start = None
        current_end = None


    for label_id, offset in zip(
        label_ids,
        offsets,
    ):

        if label_id == ignore_index:
            continue

        start, end = offset

        if start == end:
            continue


        label_name = id2label[
            label_id
        ]


        if label_name == "O":
            close_current_span()
            continue


        prefix, entity_type = (
            label_name.split(
                "-",
                1,
            )
        )


        if prefix == "B":
            close_current_span()

            current_label = entity_type
            current_start = start
            current_end = end


        elif prefix == "I":

            if (
                current_label
                == entity_type
            ):
                current_end = end

            else:
                close_current_span()

                current_label = entity_type
                current_start = start
                current_end = end


    close_current_span()

    return spans

In [50]:
validation_pred_spans = []


for record, pred_ids in zip(
    validation_records,
    validation_pred_ids,
):

    pred_spans = decode_bio_to_spans(
        label_ids=pred_ids,
        offsets=record[
            "offset_mapping"
        ],
        id2label=id2label,
        ignore_index=IGNORE_INDEX,
    )


    validation_pred_spans.append(
        pred_spans
    )


print(
    "VALIDATION SPAN RECONSTRUCTION"
)

print("=" * 60)

print(
    f"Records : "
    f"{len(validation_pred_spans):,}"
)

print(
    f"Predicted spans : "
    f"{sum(len(spans) for spans in validation_pred_spans):,}"
)

VALIDATION SPAN RECONSTRUCTION
Records : 5,163
Predicted spans : 6,432


In [51]:
validation_gold_spans = []


for record in validation_records:

    gold_spans = decode_bio_to_spans(
        label_ids=record[
            "labels"
        ],
        offsets=record[
            "offset_mapping"
        ],
        id2label=id2label,
        ignore_index=IGNORE_INDEX,
    )


    validation_gold_spans.append(
        gold_spans
    )


total_gold_spans = sum(
    len(spans)
    for spans
    in validation_gold_spans
)

total_pred_spans = sum(
    len(spans)
    for spans
    in validation_pred_spans
)


print(
    "SPAN COUNT CHECK"
)

print("=" * 60)

print(
    f"Gold spans      : "
    f"{total_gold_spans:,}"
)

print(
    f"Predicted spans : "
    f"{total_pred_spans:,}"
)

SPAN COUNT CHECK
Gold spans      : 6,432
Predicted spans : 6,432


### 31.6 원본 Character Span Gold 로드

현재까지의 Gold Span은 Tokenized Dataset의 BIO Label을 다시 Character Span으로 복원한 결과이다.

BIO Alignment 과정까지 포함한 전체 파이프라인의 정확성을 검증하기 위해, 모델의 Prediction Span을 Tokenization 이전의 `validation_blocks.jsonl`에 저장된 원본 Gold Entity Span과 직접 비교한다.

이를 통해 다음 과정 전체를 검증한다.

- Character Span → BIO Alignment
- mmBERT Prediction
- BIO Prediction → Character Span Reconstruction

In [52]:
from pathlib import Path
import json


VALIDATION_BLOCK_PATH = Path(
    "../data/processed/blocks/validation_blocks.jsonl"
)


validation_block_records = []


with VALIDATION_BLOCK_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        if line.strip():

            validation_block_records.append(
                json.loads(line)
            )


print(
    "ORIGINAL VALIDATION BLOCK DATASET"
)

print("=" * 60)

print(
    f"Blocks : "
    f"{len(validation_block_records):,}"
)

print(
    f"Entities : "
    f"{sum(len(record['entities']) for record in validation_block_records):,}"
)

print()
print(
    "Example keys:"
)

print(
    validation_block_records[0].keys()
)

print()
print(
    "Example entity:"
)

print(
    validation_block_records[0]["entities"][:2]
)

ORIGINAL VALIDATION BLOCK DATASET
Blocks : 5,163
Entities : 6,432

Example keys:
dict_keys(['log_type', 'source', 'template_family_id', 'is_synthetic', 'original_template_id', 'split', 'sample_id', 'source_sample_id', 'template_id', 'source_template_id', 'sample_type', 'source_sample_type', 'text', 'entities', 'structure_type', 'source_start', 'source_end', 'segments'])

Example entity:
[{'start': 0, 'end': 10, 'label': 'IP_ADDRESS', 'value': '192.0.2.20'}]


### 31.7 원본 Gold Character Span 정규화

`validation_blocks.jsonl`에 저장된 원본 Entity를 평가 가능한 `(start, end, label)` 형태로 변환한다.

Prediction Span과 원본 Gold Span을 직접 비교하여 Tokenization 이전의 Character Span 기준으로 정확성을 검증한다.

In [53]:
validation_original_gold_spans = []


for record in validation_block_records:

    spans = [
        {
            "start": entity["start"],
            "end": entity["end"],
            "label": entity["label"],
        }
        for entity
        in record["entities"]
    ]

    validation_original_gold_spans.append(
        spans
    )


print(
    "ORIGINAL GOLD SPAN CHECK"
)

print("=" * 60)

print(
    f"Records : "
    f"{len(validation_original_gold_spans):,}"
)

print(
    f"Spans   : "
    f"{sum(len(spans) for spans in validation_original_gold_spans):,}"
)

ORIGINAL GOLD SPAN CHECK
Records : 5,163
Spans   : 6,432


In [54]:
block_id_mismatches = 0


for block_record, token_record in zip(
    validation_block_records,
    validation_records,
):

    if (
        block_record["sample_id"]
        != token_record["block_sample_id"]
    ):
        block_id_mismatches += 1


print(
    "BLOCK ID ALIGNMENT CHECK"
)

print("=" * 60)

print(
    f"Compared records : "
    f"{len(validation_records):,}"
)

print(
    f"ID mismatches    : "
    f"{block_id_mismatches:,}"
)

BLOCK ID ALIGNMENT CHECK
Compared records : 5,163
ID mismatches    : 0


### 31.9 Exact Character Span 평가

Prediction과 원본 Gold Entity의 `start`, `end`, `label`이 모두 동일한 경우에만 True Positive로 인정한다.

이를 통해 실제 마스킹 대상 Character 구간을 정확하게 탐지했는지 평가한다.

- True Positive: Gold와 정확히 일치하는 Span
- False Positive: Gold에 존재하지 않는 Prediction Span
- False Negative: 모델이 탐지하지 못한 Gold Span

In [55]:
total_tp = 0
total_fp = 0
total_fn = 0


for gold_spans, pred_spans in zip(
    validation_original_gold_spans,
    validation_pred_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in gold_spans
    }

    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in pred_spans
    }


    total_tp += len(
        gold_set
        & pred_set
    )

    total_fp += len(
        pred_set
        - gold_set
    )

    total_fn += len(
        gold_set
        - pred_set
    )


span_precision = (
    total_tp
    / (total_tp + total_fp)
    if total_tp + total_fp > 0
    else 0.0
)

span_recall = (
    total_tp
    / (total_tp + total_fn)
    if total_tp + total_fn > 0
    else 0.0
)

span_f1 = (
    2
    * span_precision
    * span_recall
    / (
        span_precision
        + span_recall
    )
    if span_precision + span_recall > 0
    else 0.0
)


print(
    "EXACT CHARACTER SPAN RESULT"
)

print("=" * 60)

print(
    f"True Positive  : "
    f"{total_tp:,}"
)

print(
    f"False Positive : "
    f"{total_fp:,}"
)

print(
    f"False Negative : "
    f"{total_fn:,}"
)

print()

print(
    f"Precision : "
    f"{span_precision:.6f}"
)

print(
    f"Recall    : "
    f"{span_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{span_f1:.6f}"
)

EXACT CHARACTER SPAN RESULT
True Positive  : 5,430
False Positive : 1,002
False Negative : 1,002

Precision : 0.844216
Recall    : 0.844216
F1-score  : 0.844216


In [56]:
from collections import defaultdict


entity_span_counts = defaultdict(
    lambda: {
        "tp": 0,
        "fp": 0,
        "fn": 0,
    }
)


for gold_spans, pred_spans in zip(
    validation_original_gold_spans,
    validation_pred_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in gold_spans
    }

    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in pred_spans
    }


    for span in (
        gold_set
        & pred_set
    ):
        label = span[2]

        entity_span_counts[
            label
        ]["tp"] += 1


    for span in (
        pred_set
        - gold_set
    ):
        label = span[2]

        entity_span_counts[
            label
        ]["fp"] += 1


    for span in (
        gold_set
        - pred_set
    ):
        label = span[2]

        entity_span_counts[
            label
        ]["fn"] += 1

In [57]:
entity_span_rows = []


for entity in sorted(
    entity_span_counts.keys()
):

    counts = (
        entity_span_counts[
            entity
        ]
    )

    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]


    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    f1 = (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
        if precision + recall > 0
        else 0.0
    )


    entity_span_rows.append(
        {
            "entity": entity,
            "support": tp + fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    )


entity_span_df = pd.DataFrame(
    entity_span_rows
)


display(
    entity_span_df
)

,entity,support,precision,recall,f1
0,API_KEY,174,0.574713,0.574713,0.574713
1,CONNECTION_STRING,194,0.695876,0.695876,0.695876
2,EMAIL,86,0.523256,0.523256,0.523256
3,IP_ADDRESS,1993,1.000000,1.000000,1.000000
4,PARAM_VALUE,2347,0.864082,0.864082,0.864082
5,PASSWORD,213,0.755869,0.755869,0.755869
6,PERSON,90,1.000000,1.000000,1.000000
7,PHONE,25,0.360000,0.360000,0.360000
8,PRIVATE_KEY,32,0.000000,0.000000,0.000000
9,SECRET,155,0.832258,0.832258,0.832258


### 31.11 BIO Alignment Character Boundary 검증

Token-level BIO Prediction은 Validation Gold BIO Label과 완전히 일치했지만, 원본 Character Span과 비교한 Exact Span F1은 0.8442로 나타났다.

이 차이가 모델 Prediction의 오류인지, Character Span을 BIO Token Label로 변환하는 과정에서 발생한 Boundary 정보 손실인지 구분한다.

이를 위해 모델 Prediction이 아닌 Gold BIO Label 자체를 다시 Character Span으로 복원한 결과와 원본 Gold Character Span을 직접 비교한다.

만약 이 결과가 모델 Prediction의 Exact Span 성능과 동일하다면, 현재 Exact Span 성능의 한계는 모델이 아니라 Tokenization / BIO Alignment 과정에서 발생한 Boundary 정보 손실에 의해 결정된 것으로 볼 수 있다.

In [58]:
alignment_tp = 0
alignment_fp = 0
alignment_fn = 0


for original_spans, bio_spans in zip(
    validation_original_gold_spans,
    validation_gold_spans,
):

    original_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in original_spans
    }

    bio_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in bio_spans
    }


    alignment_tp += len(
        original_set
        & bio_set
    )

    alignment_fp += len(
        bio_set
        - original_set
    )

    alignment_fn += len(
        original_set
        - bio_set
    )


alignment_precision = (
    alignment_tp
    / (
        alignment_tp
        + alignment_fp
    )
)

alignment_recall = (
    alignment_tp
    / (
        alignment_tp
        + alignment_fn
    )
)

alignment_f1 = (
    2
    * alignment_precision
    * alignment_recall
    / (
        alignment_precision
        + alignment_recall
    )
)


print(
    "BIO ALIGNMENT EXACT SPAN CEILING"
)

print("=" * 60)

print(
    f"True Positive  : "
    f"{alignment_tp:,}"
)

print(
    f"False Positive : "
    f"{alignment_fp:,}"
)

print(
    f"False Negative : "
    f"{alignment_fn:,}"
)

print()

print(
    f"Precision : "
    f"{alignment_precision:.6f}"
)

print(
    f"Recall    : "
    f"{alignment_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{alignment_f1:.6f}"
)

BIO ALIGNMENT EXACT SPAN CEILING
True Positive  : 5,430
False Positive : 1,002
False Negative : 1,002

Precision : 0.844216
Recall    : 0.844216
F1-score  : 0.844216


In [59]:
boundary_errors = []

count_mismatches = 0
label_mismatches = 0


for block_record, original_spans, bio_spans in zip(
    validation_block_records,
    validation_original_gold_spans,
    validation_gold_spans,
):

    original_sorted = sorted(
        original_spans,
        key=lambda span: (
            span["start"],
            span["end"],
            span["label"],
        ),
    )

    bio_sorted = sorted(
        bio_spans,
        key=lambda span: (
            span["start"],
            span["end"],
            span["label"],
        ),
    )


    if (
        len(original_sorted)
        != len(bio_sorted)
    ):
        count_mismatches += 1
        continue


    for gold_span, decoded_span in zip(
        original_sorted,
        bio_sorted,
    ):

        if (
            gold_span["label"]
            != decoded_span["label"]
        ):
            label_mismatches += 1
            continue


        if (
            gold_span["start"]
            == decoded_span["start"]
            and
            gold_span["end"]
            == decoded_span["end"]
        ):
            continue


        text = block_record[
            "text"
        ]


        boundary_errors.append(
            {
                "sample_id":
                    block_record[
                        "sample_id"
                    ],

                "label":
                    gold_span["label"],

                "gold_start":
                    gold_span["start"],

                "gold_end":
                    gold_span["end"],

                "decoded_start":
                    decoded_span[
                        "start"
                    ],

                "decoded_end":
                    decoded_span[
                        "end"
                    ],

                "start_delta":
                    decoded_span[
                        "start"
                    ]
                    - gold_span[
                        "start"
                    ],

                "end_delta":
                    decoded_span[
                        "end"
                    ]
                    - gold_span[
                        "end"
                    ],

                "gold_value":
                    text[
                        gold_span["start"]:
                        gold_span["end"]
                    ],

                "decoded_value":
                    text[
                        decoded_span["start"]:
                        decoded_span["end"]
                    ],
            }
        )


print(
    "BOUNDARY ERROR SUMMARY"
)

print("=" * 60)

print(
    f"Boundary errors : "
    f"{len(boundary_errors):,}"
)

print(
    f"Count mismatches : "
    f"{count_mismatches:,}"
)

print(
    f"Label mismatches : "
    f"{label_mismatches:,}"
)

BOUNDARY ERROR SUMMARY
Boundary errors : 1,002
Count mismatches : 0
Label mismatches : 0


In [60]:
boundary_error_df = (
    pd.DataFrame(
        boundary_errors
    )
)


boundary_error_summary = (
    boundary_error_df
    .groupby(
        "label"
    )
    .agg(
        errors=(
            "label",
            "size",
        ),

        mean_start_delta=(
            "start_delta",
            "mean",
        ),

        mean_end_delta=(
            "end_delta",
            "mean",
        ),

        min_start_delta=(
            "start_delta",
            "min",
        ),

        max_start_delta=(
            "start_delta",
            "max",
        ),

        min_end_delta=(
            "end_delta",
            "min",
        ),

        max_end_delta=(
            "end_delta",
            "max",
        ),
    )
    .reset_index()
)


display(
    boundary_error_summary
)

,label,errors,mean_start_delta,mean_end_delta,min_start_delta,max_start_delta,min_end_delta,max_end_delta
0,API_KEY,74,-1.000000,0.000000,-1,-1,0,0
1,CONNECTION_STRING,59,-1.000000,0.000000,-1,-1,0,0
2,EMAIL,41,-1.000000,0.000000,-1,-1,0,0
3,PARAM_VALUE,319,-0.996865,0.003135,-1,0,0,1
4,PASSWORD,52,-0.865385,0.134615,-1,0,0,1
5,PHONE,16,-1.000000,0.000000,-1,-1,0,0
6,PRIVATE_KEY,32,-1.000000,0.000000,-1,-1,0,0
7,SECRET,26,-1.000000,0.000000,-1,-1,0,0
8,SESSION_ID,96,-1.000000,0.000000,-1,-1,0,0
9,TOKEN,287,-1.000000,0.000000,-1,-1,0,0


In [61]:
display(
    boundary_error_df[
        [
            "label",
            "gold_value",
            "decoded_value",
            "start_delta",
            "end_delta",
        ]
    ].head(
        20
    )
)

,label,gold_value,decoded_value,start_delta,end_delta
0,PARAM_VALUE,%EB%85%B8%ED%8A%B8%EB%B6%81,/%EB%85%B8%ED%8A%B8%EB%B6%81,-1,0
1,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,/%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
2,PARAM_VALUE,%EA%B2%B0%EC%A0%9C%20%EC%98%A4%EB%A5%98,/%EA%B2%B0%EC%A0%9C%20%EC%98%A4%EB%A5%98,-1,0
3,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,/%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
4,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,/%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
5,PARAM_VALUE,%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,-1,0
6,PARAM_VALUE,%EB%85%B8%ED%8A%B8%EB%B6%81,=%EB%85%B8%ED%8A%B8%EB%B6%81,-1,0
7,PARAM_VALUE,%EB%85%B8%ED%8A%B8%EB%B6%81,=%EB%85%B8%ED%8A%B8%EB%B6%81,-1,0
8,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,=%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
9,PARAM_VALUE,%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,-1,0


### 31.13 Character Boundary Error 패턴 분석

BIO Alignment 과정에서 발생한 Character Boundary Error의 형태를 분석한다.

각 오류에 대해 시작 위치와 종료 위치가 원본 Gold Span에서 얼마나 벗어났는지 확인하고, 추가로 포함된 Character를 분석한다.

이를 통해 Character Span 복원 과정에서 적용할 Boundary Refinement 규칙을 결정한다.

In [62]:
print(
    "BOUNDARY DELTA DISTRIBUTION"
)

print("=" * 60)

print("\nStart delta")

display(
    boundary_error_df[
        "start_delta"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "start_delta"
    )
    .reset_index(
        name="count"
    )
)


print("\nEnd delta")

display(
    boundary_error_df[
        "end_delta"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "end_delta"
    )
    .reset_index(
        name="count"
    )
)

BOUNDARY DELTA DISTRIBUTION

Start delta


,start_delta,count
0,-1,994
1,0,8



End delta


,end_delta,count
0,0,994
1,1,8


In [63]:
def get_extra_prefix(
    row,
):
    if row["decoded_start"] >= row["gold_start"]:
        return ""

    return row["decoded_value"][
        :row["gold_start"] - row["decoded_start"]
    ]


def get_extra_suffix(
    row,
):
    if row["decoded_end"] <= row["gold_end"]:
        return ""

    extra_length = (
        row["decoded_end"]
        - row["gold_end"]
    )

    return row["decoded_value"][
        -extra_length:
    ]


boundary_error_df[
    "extra_prefix"
] = boundary_error_df.apply(
    get_extra_prefix,
    axis=1,
)


boundary_error_df[
    "extra_suffix"
] = boundary_error_df.apply(
    get_extra_suffix,
    axis=1,
)


print(
    "EXTRA PREFIX CHARACTERS"
)

print("=" * 60)

display(
    boundary_error_df[
        "extra_prefix"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
    .rename_axis(
        "character"
    )
    .reset_index(
        name="count"
    )
)


print(
    "EXTRA SUFFIX CHARACTERS"
)

print("=" * 60)

display(
    boundary_error_df[
        "extra_suffix"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
    .rename_axis(
        "character"
    )
    .reset_index(
        name="count"
    )
)

EXTRA PREFIX CHARACTERS


,character,count
0,,858
1,=,80
2,/,56
3,,8


EXTRA SUFFIX CHARACTERS


,character,count
0,,994
1,"""",7
2,/,1


### 31.14 Boundary Error 추가 Character 분석

Boundary Error 대부분은 원본 Character Span보다 한 Character 넓게 복원되는 형태로 나타났다.

DataFrame 출력에서는 공백, 줄바꿈, 탭 등의 문자가 빈 값처럼 표시될 수 있으므로 `repr()`을 사용하여 실제로 추가된 Character를 확인한다.

또한 Entity Type별로 어떤 Character가 Boundary Error를 유발하는지 분석하여 후처리 규칙 적용 가능성을 검토한다.

In [64]:
prefix_repr_counts = (
    boundary_error_df[
        "extra_prefix"
    ]
    .map(repr)
    .value_counts()
    .rename_axis(
        "extra_prefix"
    )
    .reset_index(
        name="count"
    )
)


suffix_repr_counts = (
    boundary_error_df[
        "extra_suffix"
    ]
    .map(repr)
    .value_counts()
    .rename_axis(
        "extra_suffix"
    )
    .reset_index(
        name="count"
    )
)


print(
    "EXTRA PREFIX - REPR"
)

print("=" * 60)

display(
    prefix_repr_counts
)


print(
    "EXTRA SUFFIX - REPR"
)

print("=" * 60)

display(
    suffix_repr_counts
)

EXTRA PREFIX - REPR


,extra_prefix,count
0,' ',858
1,'=',80
2,'/',56
3,'',8


EXTRA SUFFIX - REPR


,extra_suffix,count
0,'',994
1,"'""'",7
2,'/',1


In [65]:
boundary_pattern_df = (
    boundary_error_df
    .assign(
        extra_prefix_repr=
            boundary_error_df[
                "extra_prefix"
            ].map(repr),

        extra_suffix_repr=
            boundary_error_df[
                "extra_suffix"
            ].map(repr),
    )
    .groupby(
        [
            "label",
            "start_delta",
            "end_delta",
            "extra_prefix_repr",
            "extra_suffix_repr",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )
)


display(
    boundary_pattern_df.head(
        50
    )
)

,label,start_delta,end_delta,extra_prefix_repr,extra_suffix_repr,count
15,TOKEN,-1,0,' ','',286
4,PARAM_VALUE,-1,0,' ','',202
13,SESSION_ID,-1,0,' ','',94
0,API_KEY,-1,0,' ','',73
6,PARAM_VALUE,-1,0,'=','',60
2,CONNECTION_STRING,-1,0,' ','',59
5,PARAM_VALUE,-1,0,'/','',56
8,PASSWORD,-1,0,' ','',45
3,EMAIL,-1,0,' ','',41
11,PRIVATE_KEY,-1,0,' ','',32


## 32. Boundary Refinement

Character Span 분석 결과, 1,002개의 Exact Span 오류 중 대부분은 Tokenizer가 민감정보와 인접한 Character를 하나의 Token으로 묶으면서 발생한 Boundary Error였다.

특히 858개 오류에서 Prediction Span의 시작 부분에 공백 한 Character가 추가되어 있었다.

공백은 민감정보 값 자체가 아니라 주변 구분자로 사용되는 경우가 일반적이므로, 첫 번째 Boundary Refinement로 Prediction Span의 시작과 끝에 포함된 whitespace를 제거한다.

이 단계에서는 `=`, `/`, `"` 등의 문자는 제거하지 않는다.

이를 통해 특정 데이터 패턴에 과도하게 의존하지 않고 일반화 가능한 최소한의 후처리만 적용했을 때 Exact Character Span 성능이 얼마나 개선되는지 확인한다.

In [66]:
def trim_span_whitespace(
    text,
    span,
):
    start = span["start"]
    end = span["end"]


    while (
        start < end
        and text[start].isspace()
    ):
        start += 1


    while (
        end > start
        and text[end - 1].isspace()
    ):
        end -= 1


    return {
        "start": start,
        "end": end,
        "label": span["label"],
    }

In [67]:
validation_refined_spans = []


for block_record, pred_spans in zip(
    validation_block_records,
    validation_pred_spans,
):

    text = block_record[
        "text"
    ]


    refined_spans = [
        trim_span_whitespace(
            text=text,
            span=span,
        )
        for span
        in pred_spans
    ]


    validation_refined_spans.append(
        refined_spans
    )


print(
    "WHITESPACE REFINEMENT"
)

print("=" * 60)

print(
    f"Records       : "
    f"{len(validation_refined_spans):,}"
)

print(
    f"Refined spans : "
    f"{sum(len(spans) for spans in validation_refined_spans):,}"
)

WHITESPACE REFINEMENT
Records       : 5,163
Refined spans : 6,432


In [68]:
refined_change_count = 0


for original_pred, refined_pred in zip(
    validation_pred_spans,
    validation_refined_spans,
):

    for before, after in zip(
        original_pred,
        refined_pred,
    ):

        if (
            before["start"]
            != after["start"]
            or
            before["end"]
            != after["end"]
        ):
            refined_change_count += 1


print(
    "REFINEMENT CHANGE CHECK"
)

print("=" * 60)

print(
    f"Changed spans : "
    f"{refined_change_count:,}"
)

REFINEMENT CHANGE CHECK
Changed spans : 858


### 32.5 Whitespace Refinement 성능 평가

Whitespace Boundary Refinement를 적용한 Prediction Span을 원본 Gold Character Span과 다시 비교한다.

`start`, `end`, `label`이 모두 동일한 경우에만 Exact Match로 인정한다.

이를 통해 단순하고 일반화 가능한 whitespace 제거만으로 Character Span 성능이 얼마나 개선되는지 평가한다.

In [69]:
refined_tp = 0
refined_fp = 0
refined_fn = 0


for gold_spans, pred_spans in zip(
    validation_original_gold_spans,
    validation_refined_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in gold_spans
    }


    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span
        in pred_spans
    }


    refined_tp += len(
        gold_set
        & pred_set
    )

    refined_fp += len(
        pred_set
        - gold_set
    )

    refined_fn += len(
        gold_set
        - pred_set
    )


refined_precision = (
    refined_tp
    / (
        refined_tp
        + refined_fp
    )
    if refined_tp + refined_fp > 0
    else 0.0
)


refined_recall = (
    refined_tp
    / (
        refined_tp
        + refined_fn
    )
    if refined_tp + refined_fn > 0
    else 0.0
)


refined_f1 = (
    2
    * refined_precision
    * refined_recall
    / (
        refined_precision
        + refined_recall
    )
    if refined_precision + refined_recall > 0
    else 0.0
)


print(
    "WHITESPACE REFINED EXACT SPAN RESULT"
)

print("=" * 60)

print(
    f"True Positive  : "
    f"{refined_tp:,}"
)

print(
    f"False Positive : "
    f"{refined_fp:,}"
)

print(
    f"False Negative : "
    f"{refined_fn:,}"
)

print()

print(
    f"Precision : "
    f"{refined_precision:.6f}"
)

print(
    f"Recall    : "
    f"{refined_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{refined_f1:.6f}"
)

WHITESPACE REFINED EXACT SPAN RESULT
True Positive  : 6,288
False Positive : 144
False Negative : 144

Precision : 0.977612
Recall    : 0.977612
F1-score  : 0.977612


In [70]:
remaining_boundary_errors = []


for (
    block_record,
    gold_spans,
    refined_spans,
) in zip(
    validation_block_records,
    validation_original_gold_spans,
    validation_refined_spans,
):

    gold_sorted = sorted(
        gold_spans,
        key=lambda span: (
            span["start"],
            span["end"],
            span["label"],
        ),
    )

    refined_sorted = sorted(
        refined_spans,
        key=lambda span: (
            span["start"],
            span["end"],
            span["label"],
        ),
    )


    for gold_span, pred_span in zip(
        gold_sorted,
        refined_sorted,
    ):

        if (
            gold_span["start"]
            == pred_span["start"]
            and
            gold_span["end"]
            == pred_span["end"]
            and
            gold_span["label"]
            == pred_span["label"]
        ):
            continue


        text = block_record[
            "text"
        ]


        remaining_boundary_errors.append(
            {
                "label":
                    gold_span["label"],

                "gold_value":
                    text[
                        gold_span["start"]:
                        gold_span["end"]
                    ],

                "pred_value":
                    text[
                        pred_span["start"]:
                        pred_span["end"]
                    ],

                "start_delta":
                    pred_span["start"]
                    - gold_span["start"],

                "end_delta":
                    pred_span["end"]
                    - gold_span["end"],
            }
        )


remaining_boundary_error_df = (
    pd.DataFrame(
        remaining_boundary_errors
    )
)


print(
    "REMAINING BOUNDARY ERRORS"
)

print("=" * 60)

print(
    f"Errors : "
    f"{len(remaining_boundary_errors):,}"
)


display(
    remaining_boundary_error_df.head(
        20
    )
)

REMAINING BOUNDARY ERRORS
Errors : 144


,label,gold_value,pred_value,start_delta,end_delta
0,PARAM_VALUE,%EB%85%B8%ED%8A%B8%EB%B6%81,/%EB%85%B8%ED%8A%B8%EB%B6%81,-1,0
1,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,/%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
2,PARAM_VALUE,%EA%B2%B0%EC%A0%9C%20%EC%98%A4%EB%A5%98,/%EA%B2%B0%EC%A0%9C%20%EC%98%A4%EB%A5%98,-1,0
3,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,/%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
4,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,/%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
5,PARAM_VALUE,%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,-1,0
6,PARAM_VALUE,%EB%85%B8%ED%8A%B8%EB%B6%81,=%EB%85%B8%ED%8A%B8%EB%B6%81,-1,0
7,PARAM_VALUE,%EB%85%B8%ED%8A%B8%EB%B6%81,=%EB%85%B8%ED%8A%B8%EB%B6%81,-1,0
8,PARAM_VALUE,%EC%95%84%EC%9D%B4%ED%8F%B0%2015,=%EC%95%84%EC%9D%B4%ED%8F%B0%2015,-1,0
9,PARAM_VALUE,%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8,-1,0


### 32.7 Remaining Boundary Error Context Analysis

Whitespace Refinement 적용 후 Exact Character Span F1은 0.9776으로 향상되었으며, 144개의 Boundary Error가 남았다.

남은 오류는 주로 `=`, `/`, `"` 등의 구분자가 Prediction Span에 한 Character 추가된 형태이다.

그러나 이러한 문자는 Token, URL, Base64 등의 실제 민감정보 값 내부에도 존재할 수 있으므로 단순히 모든 위치에서 제거할 수 없다.

따라서 Prediction Span의 앞뒤 문자와 주변 문맥을 분석하여 실제 추론 과정에서도 사용할 수 있는 일반화 가능한 Boundary Refinement 규칙을 검토한다.

In [71]:
remaining_error_details = []


for (
    block_record,
    gold_spans,
    refined_spans,
) in zip(
    validation_block_records,
    validation_original_gold_spans,
    validation_refined_spans,
):

    text = block_record["text"]


    gold_sorted = sorted(
        gold_spans,
        key=lambda span: (
            span["start"],
            span["end"],
            span["label"],
        ),
    )

    pred_sorted = sorted(
        refined_spans,
        key=lambda span: (
            span["start"],
            span["end"],
            span["label"],
        ),
    )


    for gold_span, pred_span in zip(
        gold_sorted,
        pred_sorted,
    ):

        if (
            gold_span["start"] == pred_span["start"]
            and
            gold_span["end"] == pred_span["end"]
            and
            gold_span["label"] == pred_span["label"]
        ):
            continue


        pred_start = pred_span["start"]
        pred_end = pred_span["end"]

        gold_start = gold_span["start"]
        gold_end = gold_span["end"]


        context_start = max(
            0,
            pred_start - 20,
        )

        context_end = min(
            len(text),
            pred_end + 20,
        )


        remaining_error_details.append(
            {
                "sample_id":
                    block_record["sample_id"],

                "label":
                    gold_span["label"],

                "gold_value":
                    text[
                        gold_start:
                        gold_end
                    ],

                "pred_value":
                    text[
                        pred_start:
                        pred_end
                    ],

                "start_delta":
                    pred_start
                    - gold_start,

                "end_delta":
                    pred_end
                    - gold_end,

                "char_before_pred":
                    (
                        text[pred_start - 1]
                        if pred_start > 0
                        else ""
                    ),

                "char_after_pred":
                    (
                        text[pred_end]
                        if pred_end < len(text)
                        else ""
                    ),

                "context":
                    text[
                        context_start:
                        context_end
                    ],
            }
        )


remaining_error_detail_df = pd.DataFrame(
    remaining_error_details
)


print(
    "REMAINING ERROR DETAIL"
)

print("=" * 60)

print(
    f"Errors : "
    f"{len(remaining_error_detail_df):,}"
)

REMAINING ERROR DETAIL
Errors : 144


In [72]:
remaining_entity_summary = (
    remaining_error_detail_df
    .groupby(
        "label"
    )
    .size()
    .reset_index(
        name="errors"
    )
    .sort_values(
        "errors",
        ascending=False,
    )
)


display(
    remaining_entity_summary
)

,label,errors
1,PARAM_VALUE,117
3,PHONE,16
2,PASSWORD,7
4,SESSION_ID,2
0,API_KEY,1
5,TOKEN,1


In [73]:
context_view_df = (
    remaining_error_detail_df[
        [
            "label",
            "gold_value",
            "pred_value",
            "char_before_pred",
            "char_after_pred",
            "context",
        ]
    ]
    .copy()
)


for column in [
    "gold_value",
    "pred_value",
    "char_before_pred",
    "char_after_pred",
    "context",
]:

    context_view_df[
        column
    ] = context_view_df[
        column
    ].map(repr)


display(
    context_view_df.head(
        50
    )
)

,label,gold_value,pred_value,char_before_pred,char_after_pred,context
0,PARAM_VALUE,'%EB%85%B8%ED%8A%B8%EB%B6%81','/%EB%85%B8%ED%8A%B8%EB%B6%81','s','/',"'00] ""POST /api/users/%EB%85%B8%ED%8A%B8%EB%B6..."
1,PARAM_VALUE,'%EC%95%84%EC%9D%B4%ED%8F%B0%2015','/%EC%95%84%EC%9D%B4%ED%8F%B0%2015','s',' ',"'0] ""PATCH /api/files/%EC%95%84%EC%9D%B4%ED%8F..."
2,PARAM_VALUE,'%EA%B2%B0%EC%A0%9C%20%EC%98%A4%EB%A5%98','/%EA%B2%B0%EC%A0%9C%20%EC%98%A4%EB%A5%98','s',' ',"'900] ""GET /api/files/%EA%B2%B0%EC%A0%9C%20%EC..."
3,PARAM_VALUE,'%EC%95%84%EC%9D%B4%ED%8F%B0%2015','/%EC%95%84%EC%9D%B4%ED%8F%B0%2015','s',' ',"'0] ""PATCH /api/carts/%EC%95%84%EC%9D%B4%ED%8F..."
4,PARAM_VALUE,'%EC%95%84%EC%9D%B4%ED%8F%B0%2015','/%EC%95%84%EC%9D%B4%ED%8F%B0%2015','s',' ',"'0] ""PATCH /api/files/%EC%95%84%EC%9D%B4%ED%8F..."
5,PARAM_VALUE,'%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A...,'=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8...,'e',' ','ts?page=5398.74&size=%EA%B2%80%EC%83%89%20%ED...
6,PARAM_VALUE,'%EB%85%B8%ED%8A%B8%EB%B6%81','=%EB%85%B8%ED%8A%B8%EB%B6%81','d','/','macje-napisz.html/id=%EB%85%B8%ED%8A%B8%EB%B6...
7,PARAM_VALUE,'%EB%85%B8%ED%8A%B8%EB%B6%81','=%EB%85%B8%ED%8A%B8%EB%B6%81','d',' ','T /api/users?keyword=%EB%85%B8%ED%8A%B8%EB%B6...
8,PARAM_VALUE,'%EC%95%84%EC%9D%B4%ED%8F%B0%2015','=%EC%95%84%EC%9D%B4%ED%8F%B0%2015','d','/','ignid=PENDING/gbraid=%EC%95%84%EC%9D%B4%ED%8F...
9,PARAM_VALUE,'%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A...,'=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8...,'s',' ','O/gclid=2026-12-24/s=%EA%B2%80%EC%83%89%20%ED...


In [74]:
def detect_boundary_extra(
    row,
):

    gold = row[
        "gold_value"
    ]

    pred = row[
        "pred_value"
    ]


    if (
        row["start_delta"] == -1
        and row["end_delta"] == 0
    ):
        return (
            "PREFIX",
            pred[0],
        )


    if (
        row["start_delta"] == 0
        and row["end_delta"] == 1
    ):
        return (
            "SUFFIX",
            pred[-1],
        )


    return (
        "OTHER",
        "",
    )


boundary_extras = (
    remaining_error_detail_df
    .apply(
        detect_boundary_extra,
        axis=1,
    )
)


remaining_error_detail_df[
    "error_position"
] = [
    value[0]
    for value
    in boundary_extras
]


remaining_error_detail_df[
    "extra_char"
] = [
    value[1]
    for value
    in boundary_extras
]


remaining_pattern_summary = (
    remaining_error_detail_df
    .assign(
        extra_char_repr=
            remaining_error_detail_df[
                "extra_char"
            ].map(repr)
    )
    .groupby(
        [
            "label",
            "error_position",
            "extra_char_repr",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )
)


display(
    remaining_pattern_summary
)

,label,error_position,extra_char_repr,count
2,PARAM_VALUE,PREFIX,'=',60
1,PARAM_VALUE,PREFIX,'/',56
5,PHONE,PREFIX,'=',16
4,PASSWORD,SUFFIX,"'""'",7
6,SESSION_ID,PREFIX,'=',2
0,API_KEY,PREFIX,'=',1
3,PARAM_VALUE,SUFFIX,'/',1
7,TOKEN,PREFIX,'=',1


### 32.12 Context-aware Boundary Refinement

Whitespace Refinement 이후 144개의 Boundary Error가 남았다.

남은 오류를 분석한 결과 대부분은 다음과 같은 구조적 구분자가 민감정보 Span에 포함된 경우였다.

- `key=value` 구조의 `=`
- URL Path Segment 앞의 `/`
- 따옴표로 감싼 Password 뒤의 `"`

이러한 문자를 단순히 모든 Entity에서 제거하면 실제 민감정보 값의 일부를 손상시킬 수 있다.

따라서 Entity Type과 주변 Character를 함께 고려하여 보수적인 Boundary Refinement를 적용한다.

적용 규칙은 다음과 같다.

1. Prediction이 `=`로 시작하고 바로 앞 문자가 Key를 구성할 수 있는 문자이면 `=`를 제거한다.
2. `PARAM_VALUE`가 `/%...` 형태로 시작하면 URL Path Segment 구분자인 `/`를 제거한다.
3. `PASSWORD`가 `"`로 끝나고 그 뒤가 구조적 구분자이면 종료 따옴표를 제거한다.

패턴이 한 건뿐인 `PARAM_VALUE` 뒤쪽 `/` 오류는 과도한 규칙 생성을 방지하기 위해 이번 단계에서는 수정하지 않는다.

In [75]:
def refine_span_boundaries_v2(
    text,
    span,
):
    start = span["start"]
    end = span["end"]
    label = span["label"]


    # ---------------------------------
    # 1. Leading / trailing whitespace
    # ---------------------------------

    while (
        start < end
        and text[start].isspace()
    ):
        start += 1


    while (
        end > start
        and text[end - 1].isspace()
    ):
        end -= 1


    # ---------------------------------
    # 2. key=value 형태의 '=' 제거
    # ---------------------------------

    if (
        start < end
        and text[start] == "="
        and start > 0
    ):

        previous_char = text[start - 1]

        if (
            previous_char.isalnum()
            or previous_char in "_-."
        ):
            start += 1


    # ---------------------------------
    # 3. URL path parameter의 '/' 제거
    #
    # /%EB%85...
    #   ↓
    # %EB%85...
    # ---------------------------------

    if (
        label == "PARAM_VALUE"
        and start + 1 < end
        and text[start] == "/"
        and text[start + 1] == "%"
    ):
        start += 1


    # ---------------------------------
    # 4. quoted PASSWORD의 종료 " 제거
    # ---------------------------------

    if (
        label == "PASSWORD"
        and end > start
        and text[end - 1] == '"'
    ):

        next_char = (
            text[end]
            if end < len(text)
            else ""
        )

        quote_terminators = {
            "",
            ",",
            "}",
            "]",
            ";",
            "\n",
            "\r",
            "\t",
            " ",
        }

        if next_char in quote_terminators:
            end -= 1


    return {
        "start": start,
        "end": end,
        "label": label,
    }

In [76]:
validation_refined_v2_spans = []


for block_record, pred_spans in zip(
    validation_block_records,
    validation_pred_spans,
):

    text = block_record["text"]


    refined_spans = [
        refine_span_boundaries_v2(
            text=text,
            span=span,
        )
        for span
        in pred_spans
    ]


    validation_refined_v2_spans.append(
        refined_spans
    )


print(
    "CONTEXT-AWARE REFINEMENT"
)

print("=" * 60)

print(
    f"Records : "
    f"{len(validation_refined_v2_spans):,}"
)

print(
    f"Spans   : "
    f"{sum(len(spans) for spans in validation_refined_v2_spans):,}"
)

CONTEXT-AWARE REFINEMENT
Records : 5,163
Spans   : 6,432


In [77]:
v2_tp = 0
v2_fp = 0
v2_fn = 0


for gold_spans, pred_spans in zip(
    validation_original_gold_spans,
    validation_refined_v2_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in gold_spans
    }


    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in pred_spans
    }


    v2_tp += len(
        gold_set & pred_set
    )

    v2_fp += len(
        pred_set - gold_set
    )

    v2_fn += len(
        gold_set - pred_set
    )


v2_precision = (
    v2_tp
    / (v2_tp + v2_fp)
    if v2_tp + v2_fp > 0
    else 0.0
)


v2_recall = (
    v2_tp
    / (v2_tp + v2_fn)
    if v2_tp + v2_fn > 0
    else 0.0
)


v2_f1 = (
    2
    * v2_precision
    * v2_recall
    / (
        v2_precision
        + v2_recall
    )
    if v2_precision + v2_recall > 0
    else 0.0
)


print(
    "CONTEXT-AWARE EXACT SPAN RESULT"
)

print("=" * 60)

print(
    f"True Positive  : "
    f"{v2_tp:,}"
)

print(
    f"False Positive : "
    f"{v2_fp:,}"
)

print(
    f"False Negative : "
    f"{v2_fn:,}"
)

print()

print(
    f"Precision : "
    f"{v2_precision:.6f}"
)

print(
    f"Recall    : "
    f"{v2_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{v2_f1:.6f}"
)

CONTEXT-AWARE EXACT SPAN RESULT
True Positive  : 6,431
False Positive : 1
False Negative : 1

Precision : 0.999845
Recall    : 0.999845
F1-score  : 0.999845


In [78]:
v2_remaining_errors = []


for (
    block_record,
    gold_spans,
    pred_spans,
) in zip(
    validation_block_records,
    validation_original_gold_spans,
    validation_refined_v2_spans,
):

    text = block_record["text"]


    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in gold_spans
    }


    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in pred_spans
    }


    missing = (
        gold_set - pred_set
    )

    extra = (
        pred_set - gold_set
    )


    if not missing and not extra:
        continue


    v2_remaining_errors.append(
        {
            "sample_id":
                block_record["sample_id"],

            "text":
                repr(text),

            "missing_gold":
                list(missing),

            "extra_prediction":
                list(extra),
        }
    )


print(
    "V2 REMAINING ERRORS"
)

print("=" * 60)

print(
    f"Error records : "
    f"{len(v2_remaining_errors):,}"
)


for error in v2_remaining_errors[:10]:

    print()
    print(
        "sample_id:",
        error["sample_id"],
    )

    print(
        "gold:",
        error["missing_gold"],
    )

    print(
        "prediction:",
        error["extra_prediction"],
    )

    print(
        "text:",
        error["text"],
    )

V2 REMAINING ERRORS
Error records : 1

sample_id: http_access_08330__block_00
gold: [(81, 99, 'PARAM_VALUE')]
prediction: [(81, 100, 'PARAM_VALUE')]
text: '10.153.98.221 - - [07/Dec/2025:20:41:35 +0100] "GET /domain-m-10.html/gad_source=value_8ixHjr1rnzD-/gad_campaignid=%EA%B2%80%EC%83%89%20%ED%85%8C%EC%8A%A4%ED%8A%B8/gbraid=wUAz7ITBxcJl9L/gclid=481517173/s=true HTTP/2.0" 200 21430 "https://sklep.domain.pl/domain-m-10.html" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"'


### 32.17 Path Parameter 종료 Boundary Refinement

2차 Boundary Refinement 이후 6,432개 Entity 중 1개의 Boundary Error가 남았다.

해당 오류는 URL Path Parameter가 다음과 같은 구조를 가지면서 발생하였다.

`key=value/next_key=value`

Tokenizer가 현재 Parameter Value와 다음 Parameter를 구분하는 `/`를 이전 Entity Token에 포함하면서 Prediction Span의 종료 위치가 한 Character 길게 복원되었다.

따라서 `PARAM_VALUE` Prediction이 `/`로 끝나고, 해당 `/` 바로 뒤의 문자열이 `key=` 형태의 새로운 Path Parameter 구조를 가지는 경우에만 종료 `/`를 제거한다.

단순히 모든 `/`를 제거하지 않고 후속 문맥을 확인함으로써 실제 값 내부의 `/`가 손실되는 것을 방지한다.

In [79]:
import re


def refine_span_boundaries_v3(
    text,
    span,
):
    start = span["start"]
    end = span["end"]
    label = span["label"]


    # ---------------------------------
    # 1. Leading / trailing whitespace
    # ---------------------------------

    while (
        start < end
        and text[start].isspace()
    ):
        start += 1


    while (
        end > start
        and text[end - 1].isspace()
    ):
        end -= 1


    # ---------------------------------
    # 2. key=value 형태의 '=' 제거
    # ---------------------------------

    if (
        start < end
        and text[start] == "="
        and start > 0
    ):

        previous_char = text[start - 1]

        if (
            previous_char.isalnum()
            or previous_char in "_-."
        ):
            start += 1


    # ---------------------------------
    # 3. URL Path Parameter 앞 '/' 제거
    # ---------------------------------

    if (
        label == "PARAM_VALUE"
        and start + 1 < end
        and text[start] == "/"
        and text[start + 1] == "%"
    ):
        start += 1


    # ---------------------------------
    # 4. quoted PASSWORD의 종료 " 제거
    # ---------------------------------

    if (
        label == "PASSWORD"
        and end > start
        and text[end - 1] == '"'
    ):

        next_char = (
            text[end]
            if end < len(text)
            else ""
        )

        quote_terminators = {
            "",
            ",",
            "}",
            "]",
            ";",
            "\n",
            "\r",
            "\t",
            " ",
        }

        if next_char in quote_terminators:
            end -= 1


    # ---------------------------------
    # 5. Path Parameter 사이의 '/' 제거
    #
    # value/next_key=value
    #      ↑
    # ---------------------------------

    if (
        label == "PARAM_VALUE"
        and end > start
        and text[end - 1] == "/"
    ):

        following_text = text[end:]

        if re.match(
            r"[A-Za-z_][A-Za-z0-9_.-]*=",
            following_text,
        ):
            end -= 1


    return {
        "start": start,
        "end": end,
        "label": label,
    }

In [80]:
validation_refined_v3_spans = []


for block_record, pred_spans in zip(
    validation_block_records,
    validation_pred_spans,
):

    text = block_record["text"]

    refined_spans = [
        refine_span_boundaries_v3(
            text=text,
            span=span,
        )
        for span in pred_spans
    ]

    validation_refined_v3_spans.append(
        refined_spans
    )


print(
    "V3 BOUNDARY REFINEMENT"
)

print("=" * 60)

print(
    f"Records : "
    f"{len(validation_refined_v3_spans):,}"
)

print(
    f"Spans   : "
    f"{sum(len(spans) for spans in validation_refined_v3_spans):,}"
)

V3 BOUNDARY REFINEMENT
Records : 5,163
Spans   : 6,432


In [81]:
v3_tp = 0
v3_fp = 0
v3_fn = 0


for gold_spans, pred_spans in zip(
    validation_original_gold_spans,
    validation_refined_v3_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in gold_spans
    }

    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in pred_spans
    }


    v3_tp += len(
        gold_set & pred_set
    )

    v3_fp += len(
        pred_set - gold_set
    )

    v3_fn += len(
        gold_set - pred_set
    )


v3_precision = (
    v3_tp / (v3_tp + v3_fp)
    if v3_tp + v3_fp > 0
    else 0.0
)

v3_recall = (
    v3_tp / (v3_tp + v3_fn)
    if v3_tp + v3_fn > 0
    else 0.0
)

v3_f1 = (
    2 * v3_precision * v3_recall
    / (v3_precision + v3_recall)
    if v3_precision + v3_recall > 0
    else 0.0
)


print(
    "FINAL VALIDATION EXACT SPAN RESULT"
)

print("=" * 60)

print(
    f"True Positive  : "
    f"{v3_tp:,}"
)

print(
    f"False Positive : "
    f"{v3_fp:,}"
)

print(
    f"False Negative : "
    f"{v3_fn:,}"
)

print()

print(
    f"Precision : "
    f"{v3_precision:.6f}"
)

print(
    f"Recall    : "
    f"{v3_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{v3_f1:.6f}"
)

FINAL VALIDATION EXACT SPAN RESULT
True Positive  : 6,432
False Positive : 0
False Negative : 0

Precision : 1.000000
Recall    : 1.000000
F1-score  : 1.000000


## 33. Test Dataset Evaluation

Validation Dataset을 이용하여 모델 선택과 Character Boundary Refinement 설계를 완료하였다.

최종 Validation 결과는 다음과 같다.

- BIO Token F1: 1.0000
- Raw Exact Character Span F1: 0.8442
- Whitespace Refinement F1: 0.9776
- Context-aware Boundary Refinement F1: 1.0000

Validation 분석을 통해 확정한 `refine_span_boundaries_v3()`를 최종 Boundary Refinement 방식으로 동결한다.

이후 Test Dataset에서는 모델 또는 Boundary Refinement 규칙을 변경하지 않고 동일한 Pipeline을 그대로 적용하여 일반화 성능을 평가한다.

평가 기준은 실제 마스킹 대상 구간의 정확성을 나타내는 Exact Character Span Precision, Recall, F1-score를 Primary Metric으로 사용한다.

In [82]:
TEST_BATCH_SIZE = 2


test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)


test_pred_ids = []


best_model.eval()


with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Test inference",
    ):

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )


        outputs = best_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


        batch_predictions = (
            outputs.logits
            .argmax(dim=-1)
            .cpu()
            .tolist()
        )


        batch_attention = (
            attention_mask
            .cpu()
            .tolist()
        )


        for predictions, mask in zip(
            batch_predictions,
            batch_attention,
        ):

            sequence_length = sum(mask)

            test_pred_ids.append(
                predictions[
                    :sequence_length
                ]
            )


print(
    "TEST PREDICTION"
)

print("=" * 60)

print(
    f"Test records       : "
    f"{len(test_records):,}"
)

print(
    f"Prediction records : "
    f"{len(test_pred_ids):,}"
)

Test inference:   0%|          | 0/2693 [00:00<?, ?it/s]

TEST PREDICTION
Test records       : 5,385
Prediction records : 5,385


In [83]:
test_length_errors = 0


for record, pred_ids in zip(
    test_records,
    test_pred_ids,
):

    if (
        len(record["input_ids"])
        != len(pred_ids)
    ):
        test_length_errors += 1


print(
    "TEST PREDICTION VALIDATION"
)

print("=" * 60)

print(
    f"Records       : "
    f"{len(test_records):,}"
)

print(
    f"Predictions   : "
    f"{len(test_pred_ids):,}"
)

print(
    f"Length errors : "
    f"{test_length_errors:,}"
)

TEST PREDICTION VALIDATION
Records       : 5,385
Predictions   : 5,385
Length errors : 0


In [84]:
test_gold_label_ids = []
test_pred_label_ids = []


for record, predictions in zip(
    test_records,
    test_pred_ids,
):

    for gold_id, pred_id in zip(
        record["labels"],
        predictions,
    ):

        if gold_id == IGNORE_INDEX:
            continue

        test_gold_label_ids.append(
            gold_id
        )

        test_pred_label_ids.append(
            pred_id
        )


test_token_mismatches = sum(
    gold != pred
    for gold, pred in zip(
        test_gold_label_ids,
        test_pred_label_ids,
    )
)


print(
    "TEST TOKEN RESULT"
)

print("=" * 60)

print(
    f"Evaluated tokens : "
    f"{len(test_gold_label_ids):,}"
)

print(
    f"Mismatches       : "
    f"{test_token_mismatches:,}"
)

print(
    f"Match ratio      : "
    f"{1 - test_token_mismatches / len(test_gold_label_ids):.8f}"
)

TEST TOKEN RESULT
Evaluated tokens : 774,245
Mismatches       : 31
Match ratio      : 0.99995996


In [85]:
test_pred_spans = []


for record, pred_ids in zip(
    test_records,
    test_pred_ids,
):

    spans = decode_bio_to_spans(
        label_ids=pred_ids,
        offsets=record[
            "offset_mapping"
        ],
        id2label=id2label,
        ignore_index=IGNORE_INDEX,
    )

    test_pred_spans.append(
        spans
    )


print(
    "TEST SPAN RECONSTRUCTION"
)

print("=" * 60)

print(
    f"Records         : "
    f"{len(test_pred_spans):,}"
)

print(
    f"Predicted spans : "
    f"{sum(len(spans) for spans in test_pred_spans):,}"
)

TEST SPAN RECONSTRUCTION
Records         : 5,385
Predicted spans : 6,864


In [86]:
TEST_BLOCK_PATH = Path(
    "../data/processed/blocks/test_blocks.jsonl"
)


test_block_records = []


with TEST_BLOCK_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        if line.strip():

            test_block_records.append(
                json.loads(line)
            )


test_original_gold_spans = []


for record in test_block_records:

    spans = [
        {
            "start": entity["start"],
            "end": entity["end"],
            "label": entity["label"],
        }
        for entity in record[
            "entities"
        ]
    ]

    test_original_gold_spans.append(
        spans
    )


print(
    "TEST ORIGINAL GOLD"
)

print("=" * 60)

print(
    f"Blocks   : "
    f"{len(test_block_records):,}"
)

print(
    f"Entities : "
    f"{sum(len(spans) for spans in test_original_gold_spans):,}"
)

TEST ORIGINAL GOLD
Blocks   : 5,385
Entities : 6,867


In [87]:
test_block_id_mismatches = 0


for block_record, token_record in zip(
    test_block_records,
    test_records,
):

    if (
        block_record["sample_id"]
        != token_record["block_sample_id"]
    ):
        test_block_id_mismatches += 1


print(
    "TEST BLOCK ALIGNMENT"
)

print("=" * 60)

print(
    f"Compared records : "
    f"{len(test_records):,}"
)

print(
    f"ID mismatches    : "
    f"{test_block_id_mismatches:,}"
)

TEST BLOCK ALIGNMENT
Compared records : 5,385
ID mismatches    : 0


In [88]:
test_refined_spans = []


for block_record, pred_spans in zip(
    test_block_records,
    test_pred_spans,
):

    text = block_record["text"]


    refined_spans = [
        refine_span_boundaries_v3(
            text=text,
            span=span,
        )
        for span in pred_spans
    ]


    test_refined_spans.append(
        refined_spans
    )


print(
    "TEST BOUNDARY REFINEMENT"
)

print("=" * 60)

print(
    f"Refined spans : "
    f"{sum(len(spans) for spans in test_refined_spans):,}"
)

TEST BOUNDARY REFINEMENT
Refined spans : 6,864


In [89]:
test_tp = 0
test_fp = 0
test_fn = 0


for gold_spans, pred_spans in zip(
    test_original_gold_spans,
    test_refined_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in gold_spans
    }


    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in pred_spans
    }


    test_tp += len(
        gold_set & pred_set
    )

    test_fp += len(
        pred_set - gold_set
    )

    test_fn += len(
        gold_set - pred_set
    )


test_precision = (
    test_tp
    / (test_tp + test_fp)
    if test_tp + test_fp > 0
    else 0.0
)


test_recall = (
    test_tp
    / (test_tp + test_fn)
    if test_tp + test_fn > 0
    else 0.0
)


test_f1 = (
    2
    * test_precision
    * test_recall
    / (
        test_precision
        + test_recall
    )
    if test_precision + test_recall > 0
    else 0.0
)


print(
    "FINAL TEST EXACT SPAN RESULT"
)

print("=" * 60)

print(
    f"True Positive  : "
    f"{test_tp:,}"
)

print(
    f"False Positive : "
    f"{test_fp:,}"
)

print(
    f"False Negative : "
    f"{test_fn:,}"
)

print()

print(
    f"Precision : "
    f"{test_precision:.6f}"
)

print(
    f"Recall    : "
    f"{test_recall:.6f}"
)

print(
    f"F1-score  : "
    f"{test_f1:.6f}"
)

FINAL TEST EXACT SPAN RESULT
True Positive  : 6,836
False Positive : 28
False Negative : 31

Precision : 0.995921
Recall    : 0.995486
F1-score  : 0.995703


## 34. Test Error Analysis

동결된 mmBERT 모델과 Boundary Refinement Pipeline을 Test Dataset에 적용한 결과 Exact Character Span F1-score는 0.9957로 나타났다.

전체 6,867개의 Gold Entity 중 6,836개를 정확하게 탐지했으며, 28개의 False Positive와 31개의 False Negative가 발생하였다.

Test Dataset은 독립 평가 데이터이므로 이후 오류 분석 결과를 이용하여 모델이나 Boundary Refinement 규칙을 수정하지 않는다.

오류 분석은 현재 모델의 일반화 특성과 취약한 Entity Type 및 오류 유형을 파악하기 위한 목적으로 수행한다.

In [90]:
from collections import defaultdict


test_entity_counts = defaultdict(
    lambda: {
        "tp": 0,
        "fp": 0,
        "fn": 0,
    }
)


for gold_spans, pred_spans in zip(
    test_original_gold_spans,
    test_refined_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in gold_spans
    }

    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in pred_spans
    }


    for span in gold_set & pred_set:
        test_entity_counts[
            span[2]
        ]["tp"] += 1


    for span in pred_set - gold_set:
        test_entity_counts[
            span[2]
        ]["fp"] += 1


    for span in gold_set - pred_set:
        test_entity_counts[
            span[2]
        ]["fn"] += 1

In [91]:
test_entity_rows = []


for entity in sorted(
    test_entity_counts.keys()
):

    counts = test_entity_counts[
        entity
    ]

    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]


    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )


    test_entity_rows.append(
        {
            "entity": entity,
            "support": tp + fn,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    )


test_entity_df = pd.DataFrame(
    test_entity_rows
)


display(
    test_entity_df
)

,entity,support,tp,fp,fn,precision,recall,f1
0,API_KEY,280,280,0,0,1.000000,1.000000,1.000000
1,CONNECTION_STRING,121,121,0,0,1.000000,1.000000,1.000000
2,EMAIL,211,211,0,0,1.000000,1.000000,1.000000
3,IP_ADDRESS,2252,2252,0,0,1.000000,1.000000,1.000000
4,PARAM_VALUE,2551,2547,1,4,0.999608,0.998432,0.999019
5,PASSWORD,243,243,0,0,1.000000,1.000000,1.000000
6,PERSON,223,223,0,0,1.000000,1.000000,1.000000
7,PHONE,93,66,27,27,0.709677,0.709677,0.709677
8,PRIVATE_KEY,58,58,0,0,1.000000,1.000000,1.000000
9,SECRET,130,130,0,0,1.000000,1.000000,1.000000


In [92]:
test_error_records = []


for (
    block_record,
    gold_spans,
    pred_spans,
) in zip(
    test_block_records,
    test_original_gold_spans,
    test_refined_spans,
):

    gold_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in gold_spans
    }

    pred_set = {
        (
            span["start"],
            span["end"],
            span["label"],
        )
        for span in pred_spans
    }


    false_negatives = (
        gold_set - pred_set
    )

    false_positives = (
        pred_set - gold_set
    )


    if (
        not false_negatives
        and not false_positives
    ):
        continue


    test_error_records.append(
        {
            "sample_id":
                block_record[
                    "sample_id"
                ],

            "log_type":
                block_record[
                    "log_type"
                ],

            "text":
                block_record[
                    "text"
                ],

            "false_negatives":
                false_negatives,

            "false_positives":
                false_positives,
        }
    )


print(
    "TEST ERROR SUMMARY"
)

print("=" * 60)

print(
    f"Total blocks       : "
    f"{len(test_block_records):,}"
)

print(
    f"Error blocks       : "
    f"{len(test_error_records):,}"
)

print(
    f"Error block ratio  : "
    f"{len(test_error_records) / len(test_block_records):.4%}"
)

TEST ERROR SUMMARY
Total blocks       : 5,385
Error blocks       : 31
Error block ratio  : 0.5757%


### 34.3 Test Error Type 분석

Test Dataset에서 발생한 오류는 대부분 `PHONE` Entity에 집중되어 있었다.

전체 31개의 False Negative 중 27개가 PHONE에서 발생했으며, PARAM_VALUE에서 4개가 발생하였다.

Test Dataset은 독립 평가 데이터이므로 본 분석 결과를 이용하여 모델이나 Boundary Refinement 규칙을 수정하지 않는다.

각 오류를 다음 유형으로 구분하여 현재 모델의 일반화 한계를 분석한다.

- Boundary Error: Entity Type은 맞지만 Character 시작/종료 위치가 다름
- Type Error: 동일하거나 유사한 Character Span을 탐지했지만 Entity Type이 다름
- Missed Entity: Gold Entity와 겹치는 Prediction이 없음
- Spurious Entity: 실제 Gold Entity와 대응되지 않는 Prediction

In [93]:
def spans_overlap(
    span_a,
    span_b,
):
    return (
        span_a["start"] < span_b["end"]
        and
        span_b["start"] < span_a["end"]
    )

In [94]:
test_error_details = []


for (
    block_record,
    gold_spans,
    pred_spans,
) in zip(
    test_block_records,
    test_original_gold_spans,
    test_refined_spans,
):

    text = block_record["text"]


    matched_pred_indices = set()


    for gold_span in gold_spans:

        exact_match = None

        for pred_index, pred_span in enumerate(
            pred_spans
        ):

            if (
                gold_span["start"]
                == pred_span["start"]
                and
                gold_span["end"]
                == pred_span["end"]
                and
                gold_span["label"]
                == pred_span["label"]
            ):
                exact_match = pred_index
                break


        if exact_match is not None:

            matched_pred_indices.add(
                exact_match
            )

            continue


        overlapping_predictions = []


        for pred_index, pred_span in enumerate(
            pred_spans
        ):

            if spans_overlap(
                gold_span,
                pred_span,
            ):

                overlapping_predictions.append(
                    (
                        pred_index,
                        pred_span,
                    )
                )


        if overlapping_predictions:

            pred_index, pred_span = (
                overlapping_predictions[0]
            )

            matched_pred_indices.add(
                pred_index
            )


            if (
                gold_span["label"]
                == pred_span["label"]
            ):

                error_type = (
                    "BOUNDARY_ERROR"
                )

            else:

                error_type = (
                    "TYPE_ERROR"
                )


            test_error_details.append(
                {
                    "sample_id":
                        block_record[
                            "sample_id"
                        ],

                    "log_type":
                        block_record[
                            "log_type"
                        ],

                    "error_type":
                        error_type,

                    "gold_label":
                        gold_span[
                            "label"
                        ],

                    "pred_label":
                        pred_span[
                            "label"
                        ],

                    "gold_value":
                        text[
                            gold_span["start"]:
                            gold_span["end"]
                        ],

                    "pred_value":
                        text[
                            pred_span["start"]:
                            pred_span["end"]
                        ],

                    "gold_start":
                        gold_span[
                            "start"
                        ],

                    "gold_end":
                        gold_span[
                            "end"
                        ],

                    "pred_start":
                        pred_span[
                            "start"
                        ],

                    "pred_end":
                        pred_span[
                            "end"
                        ],
                }
            )


        else:

            test_error_details.append(
                {
                    "sample_id":
                        block_record[
                            "sample_id"
                        ],

                    "log_type":
                        block_record[
                            "log_type"
                        ],

                    "error_type":
                        "MISSED_ENTITY",

                    "gold_label":
                        gold_span[
                            "label"
                        ],

                    "pred_label":
                        None,

                    "gold_value":
                        text[
                            gold_span["start"]:
                            gold_span["end"]
                        ],

                    "pred_value":
                        None,

                    "gold_start":
                        gold_span[
                            "start"
                        ],

                    "gold_end":
                        gold_span[
                            "end"
                        ],

                    "pred_start":
                        None,

                    "pred_end":
                        None,
                }
            )


    for pred_index, pred_span in enumerate(
        pred_spans
    ):

        if pred_index in matched_pred_indices:
            continue


        test_error_details.append(
            {
                "sample_id":
                    block_record[
                        "sample_id"
                    ],

                "log_type":
                    block_record[
                        "log_type"
                    ],

                "error_type":
                    "SPURIOUS_ENTITY",

                "gold_label":
                    None,

                "pred_label":
                    pred_span[
                        "label"
                    ],

                "gold_value":
                    None,

                "pred_value":
                    text[
                        pred_span["start"]:
                        pred_span["end"]
                    ],

                "gold_start":
                    None,

                "gold_end":
                    None,

                "pred_start":
                    pred_span[
                        "start"
                    ],

                "pred_end":
                    pred_span[
                        "end"
                    ],
            }
        )


test_error_detail_df = pd.DataFrame(
    test_error_details
)

In [95]:
test_error_type_summary = (
    test_error_detail_df[
        "error_type"
    ]
    .value_counts()
    .rename_axis(
        "error_type"
    )
    .reset_index(
        name="count"
    )
)


display(
    test_error_type_summary
)

,error_type,count
0,BOUNDARY_ERROR,28
1,MISSED_ENTITY,3


In [96]:
test_error_entity_summary = (
    test_error_detail_df
    .groupby(
        [
            "error_type",
            "gold_label",
            "pred_label",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False,
    )
)


display(
    test_error_entity_summary
)

,error_type,gold_label,pred_label,count
1,BOUNDARY_ERROR,PHONE,PHONE,27
2,MISSED_ENTITY,PARAM_VALUE,NaN,3
0,BOUNDARY_ERROR,PARAM_VALUE,PARAM_VALUE,1


In [97]:
phone_errors = (
    test_error_detail_df[
        (
            test_error_detail_df[
                "gold_label"
            ] == "PHONE"
        )
        |
        (
            test_error_detail_df[
                "pred_label"
            ] == "PHONE"
        )
    ]
    .copy()
)


display(
    phone_errors[
        [
            "error_type",
            "gold_label",
            "pred_label",
            "gold_value",
            "pred_value",
            "gold_start",
            "gold_end",
            "pred_start",
            "pred_end",
        ]
    ].head(
        50
    )
)

,error_type,gold_label,pred_label,gold_value,pred_value,gold_start,gold_end,pred_start,pred_end
4,BOUNDARY_ERROR,PHONE,PHONE,+82 10 5600 4352,""":""+82 10 5600 4352",206,222,203.0,222.0
5,BOUNDARY_ERROR,PHONE,PHONE,+82-10-3650-3638,""":""+82-10-3650-3638",198,214,195.0,214.0
6,BOUNDARY_ERROR,PHONE,PHONE,+82-10-4768-3025,""":""+82-10-4768-3025",198,214,195.0,214.0
7,BOUNDARY_ERROR,PHONE,PHONE,+82 10 7220 9483,""":""+82 10 7220 9483",209,225,206.0,225.0
8,BOUNDARY_ERROR,PHONE,PHONE,+82 10 1014 8808,""":""+82 10 1014 8808",207,223,204.0,223.0
9,BOUNDARY_ERROR,PHONE,PHONE,+82 10 5998 8689,""":""+82 10 5998 8689",203,219,200.0,219.0
10,BOUNDARY_ERROR,PHONE,PHONE,+82-10-4901-2654,""":""+82-10-4901-2654",205,221,202.0,221.0
11,BOUNDARY_ERROR,PHONE,PHONE,+1-422-604-5683,""":""+1-422-604-5683",210,225,207.0,225.0
12,BOUNDARY_ERROR,PHONE,PHONE,+1-438-831-3995,""":""+1-438-831-3995",215,230,212.0,230.0
13,BOUNDARY_ERROR,PHONE,PHONE,+1-227-665-6217,""":""+1-227-665-6217",214,229,211.0,229.0


### 34.7 Missed Entity 분석

Test 오류 분석 결과 31개의 False Negative 중 28개는 Entity를 올바르게 탐지했지만 Character Boundary가 일치하지 않은 오류였다.

실제로 Prediction Span과 겹치는 Entity가 존재하지 않는 `MISSED_ENTITY`는 3건이었으며, 모두 `PARAM_VALUE`에서 발생하였다.

Test Dataset은 독립 평가 데이터이므로 해당 오류를 이용해 모델이나 후처리 규칙을 수정하지 않고, 현재 모델의 일반화 한계를 파악하기 위한 목적으로만 분석한다.

In [98]:
missed_entities = (
    test_error_detail_df[
        test_error_detail_df[
            "error_type"
        ] == "MISSED_ENTITY"
    ]
    .copy()
)


display(
    missed_entities[
        [
            "sample_id",
            "log_type",
            "gold_label",
            "gold_value",
            "gold_start",
            "gold_end",
        ]
    ]
)

,sample_id,log_type,gold_label,gold_value,gold_start,gold_end
0,http_access_00899__block_00,HTTP_ACCESS,PARAM_VALUE,2026-02-03,55,65
2,http_access_02962__block_00,HTTP_ACCESS,PARAM_VALUE,ORD-284848342,54,67
3,http_access_04974__block_00,HTTP_ACCESS,PARAM_VALUE,2026-01-25,55,65


In [99]:
missed_context_rows = []


for _, row in missed_entities.iterrows():

    sample_id = row[
        "sample_id"
    ]


    block_record = next(
        record
        for record
        in test_block_records
        if record["sample_id"]
        == sample_id
    )


    text = block_record[
        "text"
    ]

    start = int(
        row["gold_start"]
    )

    end = int(
        row["gold_end"]
    )


    context_start = max(
        0,
        start - 40,
    )

    context_end = min(
        len(text),
        end + 40,
    )


    missed_context_rows.append(
        {
            "sample_id":
                sample_id,

            "log_type":
                row["log_type"],

            "gold_value":
                repr(
                    text[start:end]
                ),

            "context":
                repr(
                    text[
                        context_start:
                        context_end
                    ]
                ),
        }
    )


missed_context_df = pd.DataFrame(
    missed_context_rows
)


display(
    missed_context_df
)

,sample_id,log_type,gold_value,context
0,http_access_00899__block_00,HTTP_ACCESS,'2026-02-03',"'- - [07/Dec/2025:05:39:44 +0100] ""HEAD /2026-..."
1,http_access_02962__block_00,HTTP_ACCESS,'ORD-284848342',"'- - [07/Dec/2025:05:39:44 +0100] ""HEAD /ORD-2..."
2,http_access_04974__block_00,HTTP_ACCESS,'2026-01-25',"'- - [07/Dec/2025:05:39:44 +0100] ""HEAD /2026-..."


In [100]:
param_boundary_errors = (
    test_error_detail_df[
        (
            test_error_detail_df[
                "error_type"
            ] == "BOUNDARY_ERROR"
        )
        &
        (
            test_error_detail_df[
                "gold_label"
            ] == "PARAM_VALUE"
        )
    ]
)


display(
    param_boundary_errors[
        [
            "sample_id",
            "gold_value",
            "pred_value",
            "gold_start",
            "gold_end",
            "pred_start",
            "pred_end",
        ]
    ]
)

,sample_id,gold_value,pred_value,gold_start,gold_end,pred_start,pred_end
1,http_access_01185__block_00,value__eF_prsQkaR_,value__eF_prsQkaR_/,63,81,63.0,82.0


### 34.10 Test Error Analysis 결과

동결된 모델과 Boundary Refinement Pipeline을 Test Dataset에 적용한 결과 Exact Character Span F1-score는 0.9957로 나타났다.

전체 6,867개의 Gold Entity 중 6,836개를 정확하게 탐지했으며, 28개의 Boundary Error와 3개의 Missed Entity가 확인되었다.

오류 유형을 분석한 결과 Entity Type을 잘못 분류한 Type Error는 발생하지 않았다.

#### PHONE

PHONE에서는 93개 중 66개가 Exact Match 되었으며 27개의 Boundary Error가 발생하였다.

모든 오류에서 모델은 PHONE Entity Type과 전화번호 자체를 올바르게 탐지했지만, JSON 구조의 `":"` 문자가 Prediction Span 앞부분에 함께 포함되어 Exact Character Span 평가에서 오류로 처리되었다.

따라서 PHONE 오류는 Entity 탐지 실패보다는 Tokenizer의 Character Boundary 문제에 해당한다.

#### PARAM_VALUE

PARAM_VALUE에서는 2,551개 중 2,547개가 정확하게 탐지되었다.

1건은 URL Path Separator `/`가 Prediction Span의 마지막 Character에 추가된 Boundary Error였다.

나머지 3건은 실제 Missed Entity였으며 다음과 같은 HTTP URL Path Segment에서 발생하였다.

- 날짜 형태의 값
- 주문 번호 형태의 값

이들은 `key=value` 또는 URL Encoding과 같은 명확한 구조적 단서 없이 URL Path 내부에 직접 포함되어 있어 일반 문자열과 구분하기 어려운 형태였다.

Test Dataset은 독립 평가 데이터이므로 이러한 오류를 이용해 모델이나 Boundary Refinement 규칙을 추가로 수정하지 않는다.

## 35. Generalization & Shortcut Analysis

독립 Test Dataset에서 Exact Character Span F1-score 0.9957을 기록하였다.

이는 매우 높은 성능이므로 단순히 모델 성능이 우수하다고 결론내리기보다, Train과 Test 사이의 데이터 중복이나 구조적 Shortcut 가능성을 추가로 검증한다.

특히 다음 항목을 분석한다.

1. Train / Test Entity Value 중복
2. Train / Test Template Family Leakage
3. Entity별 Value 중복률
4. Test의 Synthetic / Real-source별 성능
5. Log Type별 Test 성능
6. Entity 주변 Context Pattern의 Train / Test 유사성

이 분석은 Test 성능을 개선하기 위한 것이 아니라, 현재 성능이 어느 정도의 일반화 능력을 나타내는지 해석하기 위한 목적으로 수행한다.

In [101]:
TRAIN_BLOCK_PATH = Path(
    "../data/processed/blocks/train_blocks.jsonl"
)


train_block_records = []


with TRAIN_BLOCK_PATH.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        if line.strip():

            train_block_records.append(
                json.loads(line)
            )


print(
    "TRAIN BLOCK DATASET"
)

print("=" * 60)

print(
    f"Blocks   : "
    f"{len(train_block_records):,}"
)

print(
    f"Entities : "
    f"{sum(len(record['entities']) for record in train_block_records):,}"
)

TRAIN BLOCK DATASET
Blocks   : 44,626
Entities : 55,398


### 35.2 Train / Test Entity Value 중복 분석

Train과 Test에 동일한 민감정보 값이 반복적으로 등장할 경우, 모델이 Entity의 문맥이나 구조를 학습한 것이 아니라 특정 값을 암기했을 가능성이 있다.

따라서 Entity Type과 실제 Value를 함께 사용하여 Train과 Test 사이의 Exact Value Duplicate 비율을 계산한다.

In [102]:
def collect_entity_values(
    records,
):
    rows = []

    for record in records:

        for entity in record["entities"]:

            rows.append(
                {
                    "label":
                        entity["label"],

                    "value":
                        entity["value"],

                    "sample_id":
                        record["sample_id"],
                }
            )

    return pd.DataFrame(rows)


train_entity_df = collect_entity_values(
    train_block_records
)

test_entity_value_df = collect_entity_values(
    test_block_records
)


print(
    "ENTITY VALUE DATA"
)

print("=" * 60)

print(
    f"Train entities : "
    f"{len(train_entity_df):,}"
)

print(
    f"Test entities  : "
    f"{len(test_entity_value_df):,}"
)

ENTITY VALUE DATA
Train entities : 55,398
Test entities  : 6,867


In [103]:
train_entity_value_set = set(
    zip(
        train_entity_df["label"],
        train_entity_df["value"],
    )
)


test_entity_value_df[
    "seen_in_train"
] = [
    (
        label,
        value,
    )
    in train_entity_value_set

    for label, value in zip(
        test_entity_value_df["label"],
        test_entity_value_df["value"],
    )
]


duplicate_count = (
    test_entity_value_df[
        "seen_in_train"
    ]
    .sum()
)


duplicate_ratio = (
    duplicate_count
    / len(test_entity_value_df)
)


print(
    "TRAIN / TEST ENTITY VALUE DUPLICATION"
)

print("=" * 60)

print(
    f"Test entities       : "
    f"{len(test_entity_value_df):,}"
)

print(
    f"Seen in Train       : "
    f"{duplicate_count:,}"
)

print(
    f"Unseen in Train     : "
    f"{len(test_entity_value_df) - duplicate_count:,}"
)

print(
    f"Duplicate ratio     : "
    f"{duplicate_ratio:.4%}"
)

TRAIN / TEST ENTITY VALUE DUPLICATION
Test entities       : 6,867
Seen in Train       : 2,081
Unseen in Train     : 4,786
Duplicate ratio     : 30.3044%


In [104]:
entity_duplicate_summary = (
    test_entity_value_df
    .groupby(
        "label"
    )
    .agg(
        test_entities=(
            "value",
            "size",
        ),

        seen_in_train=(
            "seen_in_train",
            "sum",
        ),
    )
    .reset_index()
)


entity_duplicate_summary[
    "duplicate_ratio"
] = (
    entity_duplicate_summary[
        "seen_in_train"
    ]
    /
    entity_duplicate_summary[
        "test_entities"
    ]
)


entity_duplicate_summary = (
    entity_duplicate_summary
    .sort_values(
        "duplicate_ratio",
        ascending=False,
    )
)


display(
    entity_duplicate_summary
)

,label,test_entities,seen_in_train,duplicate_ratio
6,PERSON,223,184,0.825112
4,PARAM_VALUE,2551,1277,0.500588
2,EMAIL,211,95,0.450237
3,IP_ADDRESS,2252,525,0.233126
0,API_KEY,280,0,0.000000
1,CONNECTION_STRING,121,0,0.000000
5,PASSWORD,243,0,0.000000
7,PHONE,93,0,0.000000
8,PRIVATE_KEY,58,0,0.000000
9,SECRET,130,0,0.000000


In [105]:
test_unique_entity_values = (
    test_entity_value_df[
        [
            "label",
            "value",
        ]
    ]
    .drop_duplicates()
    .copy()
)


test_unique_entity_values[
    "seen_in_train"
] = [
    (
        label,
        value,
    )
    in train_entity_value_set

    for label, value in zip(
        test_unique_entity_values[
            "label"
        ],
        test_unique_entity_values[
            "value"
        ],
    )
]


unique_duplicate_count = (
    test_unique_entity_values[
        "seen_in_train"
    ]
    .sum()
)


print(
    "UNIQUE ENTITY VALUE DUPLICATION"
)

print("=" * 60)

print(
    f"Test unique values   : "
    f"{len(test_unique_entity_values):,}"
)

print(
    f"Seen in Train        : "
    f"{unique_duplicate_count:,}"
)

print(
    f"Unseen in Train      : "
    f"{len(test_unique_entity_values) - unique_duplicate_count:,}"
)

print(
    f"Duplicate ratio      : "
    f"{unique_duplicate_count / len(test_unique_entity_values):.4%}"
)

UNIQUE ENTITY VALUE DUPLICATION
Test unique values   : 5,498
Seen in Train        : 718
Unseen in Train      : 4,780
Duplicate ratio      : 13.0593%


### 35.6 Template Family Leakage 재검증

Train과 Test에 동일한 `template_family_id`가 포함되면 구조적으로 거의 동일한 문장이 양쪽 Dataset에 존재할 수 있다.

따라서 Train과 Test의 Template Family 집합을 다시 비교하여 직접적인 Template Family Leakage 여부를 확인한다.

In [106]:
train_template_families = {
    record["template_family_id"]
    for record
    in train_block_records
}


test_template_families = {
    record["template_family_id"]
    for record
    in test_block_records
}


template_family_overlap = (
    train_template_families
    &
    test_template_families
)


print(
    "TEMPLATE FAMILY LEAKAGE CHECK"
)

print("=" * 60)

print(
    f"Train families   : "
    f"{len(train_template_families):,}"
)

print(
    f"Test families    : "
    f"{len(test_template_families):,}"
)

print(
    f"Overlap families : "
    f"{len(template_family_overlap):,}"
)

TEMPLATE FAMILY LEAKAGE CHECK
Train families   : 5,486
Test families    : 679
Overlap families : 0


In [107]:
train_text_set = {
    record["text"]
    for record
    in train_block_records
}


test_exact_text_duplicates = sum(
    record["text"]
    in train_text_set

    for record
    in test_block_records
)


print(
    "EXACT TEXT DUPLICATION"
)

print("=" * 60)

print(
    f"Test blocks            : "
    f"{len(test_block_records):,}"
)

print(
    f"Exact text duplicates  : "
    f"{test_exact_text_duplicates:,}"
)

print(
    f"Duplicate ratio        : "
    f"{test_exact_text_duplicates / len(test_block_records):.4%}"
)

EXACT TEXT DUPLICATION
Test blocks            : 5,385
Exact text duplicates  : 73
Duplicate ratio        : 1.3556%


## 35. Synthetic Data Shortcut Audit

독립 Test Dataset에서 Exact Character Span F1-score 0.9957을 기록했지만, 데이터 생성 과정에서 합성 템플릿과 Entity별 Value Generator를 적극적으로 사용했기 때문에 모델이 실제 민감정보의 의미보다 생성 규칙에 의존했을 가능성이 있다.

특히 다음과 같은 Shortcut 가능성을 검증한다.

1. 특정 Entity가 Synthetic Data에 과도하게 의존하는지
2. Entity 주변의 Key / 문맥이 Label을 거의 결정하는지
3. Train과 Test가 서로 다른 Template Family이더라도 동일한 주변 문맥 패턴을 공유하는지
4. Entity Value 자체의 형태가 Label을 쉽게 구분하게 만드는지
5. 실제 공개 로그 기반 데이터와 합성 데이터 사이의 난이도 차이가 존재하는지

이 분석 결과는 모델이나 Test Pipeline을 수정하는 데 사용하지 않고, 현재 성능의 일반화 범위를 해석하기 위한 목적으로 사용한다.

In [108]:
train_source_summary = (
    pd.DataFrame(
        [
            {
                "sample_id": record["sample_id"],
                "source": record["source"],
                "log_type": record["log_type"],
                "is_synthetic": record["is_synthetic"],
                "entity_count": len(record["entities"]),
            }
            for record in train_block_records
        ]
    )
)


print(
    "TRAIN SYNTHETIC DATA SUMMARY"
)

print("=" * 60)

display(
    train_source_summary[
        "is_synthetic"
    ]
    .value_counts()
    .rename_axis(
        "is_synthetic"
    )
    .reset_index(
        name="blocks"
    )
)


print(
    "ENTITY COUNT BY SYNTHETIC TYPE"
)

print("=" * 60)

display(
    train_source_summary
    .groupby(
        "is_synthetic"
    )
    .agg(
        blocks=(
            "sample_id",
            "size",
        ),

        entities=(
            "entity_count",
            "sum",
        ),

        mean_entities=(
            "entity_count",
            "mean",
        ),
    )
    .reset_index()
)

TRAIN SYNTHETIC DATA SUMMARY


,is_synthetic,blocks
0,True,44626


ENTITY COUNT BY SYNTHETIC TYPE


,is_synthetic,blocks,entities,mean_entities
0,True,44626,55398,1.241384


In [ ]:
train_entity_source_rows = []


for record in train_block_records:

    for entity in record["entities"]:

        train_entity_source_rows.append(
            {
                "label":
                    entity["label"],

                "value":
                    entity["value"],

                "is_synthetic":
                    record["is_synthetic"],

                "source":
                    record["source"],

                "log_type":
                    record["log_type"],
            }
        )


train_entity_source_df = pd.DataFrame(
    train_entity_source_rows
)


entity_synthetic_summary = (
    train_entity_source_df
    .groupby(
        [
            "label",
            "is_synthetic",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)


entity_synthetic_pivot = (
    entity_synthetic_summary
    .pivot(
        index="label",
        columns="is_synthetic",
        values="count",
    )
    .fillna(0)
)


entity_synthetic_pivot.columns = [
    str(column)
    for column
    in entity_synthetic_pivot.columns
]


for column in [
    "False",
    "True",
]:

    if column not in entity_synthetic_pivot.columns:
        entity_synthetic_pivot[column] = 0


entity_synthetic_pivot = (
    entity_synthetic_pivot
    .rename(
        columns={
            "False":
                "real_source_entities",

            "True":
                "synthetic_entities",
        }
    )
)


entity_synthetic_pivot[
    "total"
] = (
    entity_synthetic_pivot[
        "real_source_entities"
    ]
    +
    entity_synthetic_pivot[
        "synthetic_entities"
    ]
)


entity_synthetic_pivot[
    "synthetic_ratio"
] = (
    entity_synthetic_pivot[
        "synthetic_entities"
    ]
    /
    entity_synthetic_pivot[
        "total"
    ]
)


entity_synthetic_pivot = (
    entity_synthetic_pivot
    .reset_index()
    .sort_values(
        "synthetic_ratio",
        ascending=False,
    )
)


display(
    entity_synthetic_pivot
)

,label,synthetic_entities,real_source_entities,total,synthetic_ratio
0,API_KEY,2969,0,2969,1.0
1,CONNECTION_STRING,1325,0,1325,1.0
2,EMAIL,1994,0,1994,1.0
3,IP_ADDRESS,16506,0,16506,1.0
4,PARAM_VALUE,18280,0,18280,1.0
5,PASSWORD,1490,0,1490,1.0
6,PERSON,1958,0,1958,1.0
7,PHONE,913,0,913,1.0
8,PRIVATE_KEY,1141,0,1141,1.0
9,SECRET,2170,0,2170,1.0


### 35.3 Entity 주변 Context Pattern 분석

합성 데이터에서는 Entity가 특정 Key 또는 고정 문맥 뒤에 반복적으로 삽입될 수 있다.

예를 들어 `password=`, `Authorization: Bearer`, `api_key=` 등의 Context가 특정 Label과 강하게 결합되어 있다면 모델은 Entity Value 자체보다 주변 Keyword를 Shortcut으로 사용할 수 있다.

각 Entity의 앞뒤 Context를 추출하여 이러한 반복성을 분석한다.

In [110]:
def collect_entity_contexts(
    records,
    left_chars=30,
    right_chars=15,
):

    rows = []


    for record in records:

        text = record["text"]


        for entity in record["entities"]:

            start = entity["start"]
            end = entity["end"]


            left_context = text[
                max(
                    0,
                    start - left_chars,
                ):
                start
            ]


            right_context = text[
                end:
                min(
                    len(text),
                    end + right_chars,
                )
            ]


            rows.append(
                {
                    "label":
                        entity["label"],

                    "value":
                        entity["value"],

                    "left_context":
                        left_context,

                    "right_context":
                        right_context,

                    "is_synthetic":
                        record["is_synthetic"],

                    "source":
                        record["source"],

                    "log_type":
                        record["log_type"],
                }
            )


    return pd.DataFrame(
        rows
    )


train_context_df = (
    collect_entity_contexts(
        train_block_records
    )
)


test_context_df = (
    collect_entity_contexts(
        test_block_records
    )
)


print(
    "ENTITY CONTEXT DATA"
)

print("=" * 60)

print(
    f"Train contexts : "
    f"{len(train_context_df):,}"
)

print(
    f"Test contexts  : "
    f"{len(test_context_df):,}"
)

ENTITY CONTEXT DATA
Train contexts : 55,398
Test contexts  : 6,867


In [112]:
import re


def normalize_context(
    text,
):

    text = text.lower()


    # 긴 숫자 / 일반 숫자
    text = re.sub(
        r"\d+",
        "<NUM>",
        text,
    )


    # 연속 whitespace
    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()


train_context_df[
    "left_context_norm"
] = (
    train_context_df[
        "left_context"
    ]
    .map(
        normalize_context
    )
)


test_context_df[
    "left_context_norm"
] = (
    test_context_df[
        "left_context"
    ]
    .map(
        normalize_context
    )
)

In [113]:
context_pattern_counts = (
    train_context_df
    .groupby(
        [
            "label",
            "left_context_norm",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)


context_pattern_counts[
    "label_total"
] = (
    context_pattern_counts
    .groupby(
        "label"
    )[
        "count"
    ]
    .transform(
        "sum"
    )
)


context_pattern_counts[
    "ratio"
] = (
    context_pattern_counts[
        "count"
    ]
    /
    context_pattern_counts[
        "label_total"
    ]
)


top_context_patterns = (
    context_pattern_counts
    .sort_values(
        [
            "label",
            "count",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "label"
    )
    .head(
        5
    )
)


display(
    top_context_patterns[
        [
            "label",
            "left_context_norm",
            "count",
            "ratio",
        ]
    ]
)

,label,left_context_norm,count,ratio
16,API_KEY,"vent"":""api_request"",""apikey"":""",495,0.166723
11,API_KEY,"nt"":""/external/api"",""apikey"":""",450,0.151566
6,API_KEY,external.api-key=,270,0.090940
7,API_KEY,external_api_key=,270,0.090940
8,API_KEY,ityexception: invalid api key,232,0.078141
20,CONNECTION_STRING,"nnection"",""connectionstring"":""",450,0.339623
17,CONNECTION_STRING,atabase: connection-string:,270,0.203774
21,CONNECTION_STRING,tion: failed to connect using,245,0.184906
18,CONNECTION_STRING,ction failed connectionstring=,180,0.135849
19,CONNECTION_STRING,database_url=,180,0.135849


In [114]:
context_diversity_rows = []


for label, group in (
    train_context_df
    .groupby(
        "label"
    )
):

    total = len(
        group
    )

    unique_contexts = (
        group[
            "left_context_norm"
        ]
        .nunique()
    )


    context_counts = (
        group[
            "left_context_norm"
        ]
        .value_counts()
    )


    top1_ratio = (
        context_counts.iloc[0]
        / total
        if total > 0
        else 0.0
    )


    top5_ratio = (
        context_counts.head(
            5
        ).sum()
        / total
        if total > 0
        else 0.0
    )


    context_diversity_rows.append(
        {
            "label":
                label,

            "entities":
                total,

            "unique_contexts":
                unique_contexts,

            "unique_context_ratio":
                unique_contexts
                / total,

            "top1_context_ratio":
                top1_ratio,

            "top5_context_ratio":
                top5_ratio,
        }
    )


context_diversity_df = (
    pd.DataFrame(
        context_diversity_rows
    )
    .sort_values(
        "top5_context_ratio",
        ascending=False,
    )
)


display(
    context_diversity_df
)

,label,entities,unique_contexts,unique_context_ratio,top1_context_ratio,top5_context_ratio
1,CONNECTION_STRING,1325,5,0.003774,0.339623,1.000000
8,PRIVATE_KEY,1141,8,0.007011,0.237511,0.737073
11,TOKEN,3097,726,0.234420,0.407168,0.728770
0,API_KEY,2969,17,0.005726,0.166723,0.578309
9,SECRET,2170,430,0.198157,0.228111,0.576037
3,IP_ADDRESS,16506,3598,0.217981,0.437659,0.555980
6,PERSON,1958,171,0.087334,0.252809,0.553115
5,PASSWORD,1490,276,0.185235,0.161074,0.485235
10,SESSION_ID,3555,1442,0.405626,0.117862,0.408158
2,EMAIL,1994,805,0.403711,0.117352,0.170010


In [115]:
train_context_pattern_set = set(
    zip(
        train_context_df[
            "label"
        ],
        train_context_df[
            "left_context_norm"
        ],
    )
)


test_context_df[
    "context_seen_in_train"
] = [
    (
        label,
        context,
    )
    in train_context_pattern_set

    for label, context in zip(
        test_context_df[
            "label"
        ],
        test_context_df[
            "left_context_norm"
        ],
    )
]


context_overlap_count = (
    test_context_df[
        "context_seen_in_train"
    ]
    .sum()
)


print(
    "TRAIN / TEST CONTEXT PATTERN OVERLAP"
)

print("=" * 60)

print(
    f"Test entities           : "
    f"{len(test_context_df):,}"
)

print(
    f"Context seen in Train   : "
    f"{context_overlap_count:,}"
)

print(
    f"Unseen context          : "
    f"{len(test_context_df) - context_overlap_count:,}"
)

print(
    f"Context overlap ratio   : "
    f"{context_overlap_count / len(test_context_df):.4%}"
)

TRAIN / TEST CONTEXT PATTERN OVERLAP
Test entities           : 6,867
Context seen in Train   : 4,681
Unseen context          : 2,186
Context overlap ratio   : 68.1666%


In [116]:
context_overlap_by_entity = (
    test_context_df
    .groupby(
        "label"
    )
    .agg(
        test_entities=(
            "value",
            "size",
        ),

        seen_context=(
            "context_seen_in_train",
            "sum",
        ),
    )
    .reset_index()
)


context_overlap_by_entity[
    "context_overlap_ratio"
] = (
    context_overlap_by_entity[
        "seen_context"
    ]
    /
    context_overlap_by_entity[
        "test_entities"
    ]
)


context_overlap_by_entity = (
    context_overlap_by_entity
    .sort_values(
        "context_overlap_ratio",
        ascending=False,
    )
)


display(
    context_overlap_by_entity
)

,label,test_entities,seen_context,context_overlap_ratio
0,API_KEY,280,280,1.000000
1,CONNECTION_STRING,121,121,1.000000
8,PRIVATE_KEY,58,58,1.000000
6,PERSON,223,203,0.910314
5,PASSWORD,243,212,0.872428
3,IP_ADDRESS,2252,1845,0.819272
11,TOKEN,276,201,0.728261
10,SESSION_ID,429,265,0.617716
2,EMAIL,211,121,0.573460
9,SECRET,130,70,0.538462


### 35.8 Value-only / Context-only Shortcut Probe

현재 Dataset의 모든 민감정보 Entity가 Synthetic Injection을 통해 생성된 것으로 확인되었다.

또한 일부 Entity에서는 소수의 주변 Context가 전체 학습 사례의 대부분을 차지하고 있으며, Train과 Test 사이에서도 동일한 Context Pattern이 높은 비율로 반복되었다.

따라서 Dataset 자체만으로 Entity Label을 쉽게 추론할 수 있는 Shortcut이 존재하는지 정량적으로 검증한다.

두 개의 단순한 Probe 모델을 학습한다.

#### Value-only Probe

Entity Value 문자열만 이용하여 Entity Type을 예측한다.

이 모델의 성능이 높다면 Entity별 Value Generator가 Label을 쉽게 구분할 수 있는 형태적 특징을 생성하고 있음을 의미한다.

#### Context-only Probe

Entity Value를 제거하고 Entity 주변 Context만 이용하여 Entity Type을 예측한다.

이 모델의 성능이 높다면 `password=`, `Authorization: Bearer`, `api_key=`와 같은 주변 구조가 Label을 직접적으로 노출하는 Context Shortcut이 존재함을 의미한다.

이 실험은 mmBERT 성능을 개선하기 위한 목적이 아니라 현재 Synthetic Dataset의 난이도와 Shortcut 정도를 평가하기 위한 진단 실험이다.

In [117]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)


value_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=30000,
)


X_train_value = value_vectorizer.fit_transform(
    train_context_df["value"].astype(str)
)

X_test_value = value_vectorizer.transform(
    test_context_df["value"].astype(str)
)


y_train_value = train_context_df["label"]
y_test_value = test_context_df["label"]


value_probe = LinearSVC(
    class_weight="balanced",
    random_state=42,
)


value_probe.fit(
    X_train_value,
    y_train_value,
)


value_pred = value_probe.predict(
    X_test_value
)

In [118]:
value_accuracy = accuracy_score(
    y_test_value,
    value_pred,
)

value_macro_f1 = f1_score(
    y_test_value,
    value_pred,
    average="macro",
)

value_weighted_f1 = f1_score(
    y_test_value,
    value_pred,
    average="weighted",
)


print(
    "VALUE-ONLY SHORTCUT PROBE"
)

print("=" * 60)

print(
    f"Accuracy    : "
    f"{value_accuracy:.6f}"
)

print(
    f"Macro F1    : "
    f"{value_macro_f1:.6f}"
)

print(
    f"Weighted F1 : "
    f"{value_weighted_f1:.6f}"
)

print()

print(
    classification_report(
        y_test_value,
        value_pred,
        digits=4,
        zero_division=0,
    )
)

VALUE-ONLY SHORTCUT PROBE
Accuracy    : 0.881171
Macro F1    : 0.794867
Weighted F1 : 0.884336

                   precision    recall  f1-score   support

          API_KEY     0.5371    0.6464    0.5867       280
CONNECTION_STRING     1.0000    1.0000    1.0000       121
            EMAIL     1.0000    1.0000    1.0000       211
       IP_ADDRESS     0.9987    1.0000    0.9993      2252
      PARAM_VALUE     0.9637    0.8953    0.9283      2551
         PASSWORD     0.4863    0.3663    0.4178       243
           PERSON     0.8229    1.0000    0.9028       223
            PHONE     0.9688    1.0000    0.9841        93
      PRIVATE_KEY     1.0000    1.0000    1.0000        58
           SECRET     0.3194    0.5308    0.3988       130
       SESSION_ID     0.5841    0.5828    0.5834       429
            TOKEN     0.6854    0.7971    0.7370       276

         accuracy                         0.8812      6867
        macro avg     0.7805    0.8182    0.7949      6867
     weighted avg

In [119]:
def build_context_only_text(
    dataframe,
):

    return (
        dataframe["left_context"]
        .fillna("")
        .astype(str)
        +
        " [ENTITY] "
        +
        dataframe["right_context"]
        .fillna("")
        .astype(str)
    )


train_context_only_text = (
    build_context_only_text(
        train_context_df
    )
)


test_context_only_text = (
    build_context_only_text(
        test_context_df
    )
)

In [120]:
context_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=30000,
)


X_train_context = (
    context_vectorizer
    .fit_transform(
        train_context_only_text
    )
)


X_test_context = (
    context_vectorizer
    .transform(
        test_context_only_text
    )
)


y_train_context = (
    train_context_df["label"]
)

y_test_context = (
    test_context_df["label"]
)


context_probe = LinearSVC(
    class_weight="balanced",
    random_state=42,
)


context_probe.fit(
    X_train_context,
    y_train_context,
)


context_pred = (
    context_probe.predict(
        X_test_context
    )
)

In [121]:
context_accuracy = accuracy_score(
    y_test_context,
    context_pred,
)

context_macro_f1 = f1_score(
    y_test_context,
    context_pred,
    average="macro",
)

context_weighted_f1 = f1_score(
    y_test_context,
    context_pred,
    average="weighted",
)


print(
    "CONTEXT-ONLY SHORTCUT PROBE"
)

print("=" * 60)

print(
    f"Accuracy    : "
    f"{context_accuracy:.6f}"
)

print(
    f"Macro F1    : "
    f"{context_macro_f1:.6f}"
)

print(
    f"Weighted F1 : "
    f"{context_weighted_f1:.6f}"
)

print()

print(
    classification_report(
        y_test_context,
        context_pred,
        digits=4,
        zero_division=0,
    )
)

CONTEXT-ONLY SHORTCUT PROBE
Accuracy    : 1.000000
Macro F1    : 1.000000
Weighted F1 : 1.000000

                   precision    recall  f1-score   support

          API_KEY     1.0000    1.0000    1.0000       280
CONNECTION_STRING     1.0000    1.0000    1.0000       121
            EMAIL     1.0000    1.0000    1.0000       211
       IP_ADDRESS     1.0000    1.0000    1.0000      2252
      PARAM_VALUE     1.0000    1.0000    1.0000      2551
         PASSWORD     1.0000    1.0000    1.0000       243
           PERSON     1.0000    1.0000    1.0000       223
            PHONE     1.0000    1.0000    1.0000        93
      PRIVATE_KEY     1.0000    1.0000    1.0000        58
           SECRET     1.0000    1.0000    1.0000       130
       SESSION_ID     1.0000    1.0000    1.0000       429
            TOKEN     1.0000    1.0000    1.0000       276

         accuracy                         1.0000      6867
        macro avg     1.0000    1.0000    1.0000      6867
     weighted a

### 35.11 Synthetic Shortcut Audit 결과

Test Dataset에서 Exact Character Span F1-score 0.9957이라는 매우 높은 성능이 확인되어, Synthetic Dataset 생성 과정에서 발생할 수 있는 Shortcut을 추가 분석하였다.

분석 결과 모든 12개 Entity Type의 학습 Entity가 Synthetic Injection을 통해 생성된 것으로 확인되었다.

또한 Entity Value만을 입력으로 사용하는 간단한 Character N-gram 기반 Linear SVM에서도 Macro F1-score 0.7949를 기록하였다.

특히 EMAIL, IP_ADDRESS, CONNECTION_STRING, PHONE, PRIVATE_KEY와 같이 일정한 형식을 가지는 Entity는 Value 자체만으로도 매우 높은 분류 성능을 보였다.

더 중요한 결과는 Entity Value를 완전히 제거하고 주변 Context만 사용하는 Context-only Probe에서 Accuracy와 Macro F1-score 모두 1.0000을 기록했다는 점이다.

이는 현재 데이터 생성 과정에서 `Authorization: Bearer`, `password=`, `apiKey`, `connectionString`과 같은 주변 문맥이 Entity Type과 지나치게 강하게 결합되어 있음을 의미한다.

Context-only Probe는 Gold Entity 위치를 사용하므로 실제 End-to-End Span Detection 문제와 동일하지는 않지만, Entity 주변 Context에 매우 강한 Label Shortcut이 존재한다는 것을 확인하는 진단 결과로 볼 수 있다.

따라서 현재 Test F1-score 0.9957은 실제 다양한 로그 환경에 대한 일반화 성능이라기보다 기존 Synthetic Data Generation Distribution 내부에서의 성능으로 해석한다.

이 문제를 개선하기 위해 Synthetic Data 생성 전략을 재설계하고, 기존 Generator와 다른 구조를 사용하는 Out-of-Distribution 평가 데이터를 별도로 구성한다.